# Build drill-run historical summary

Deterministic, synthetic-only transformation for release-scoped `v1.1.0` (explicit)
and `v1.2.0` (default) inputs, with release-gated Delta staging and one
atomic active-release pointer.


In [ ]:
# Notebook parameters cell: Fabric UI/Job Scheduler/API can override defaults at run time.
RELEASE_VERSION = "v1.2.0"
ACTIVATE_RELEASE_POINTER = True
CONTROL_ONLY_PROMOTION = True
# Recovery remains default-off; set to True only for one bounded operator-reviewed run.
RECOVERY_MODE = False
RECOVERY_OPERATOR = ""
RECOVERY_REASON = ""
RECOVERY_EXPECTED_DATASET_NAME = ""
RECOVERY_EXPECTED_RELEASE_ID = ""
RECOVERY_EXPECTED_ATTEMPT_ID = ""
RECOVERY_EXPECTED_APPLICATION_ID = ""
RECOVERY_EXPECTED_JOB_INSTANCE_ID = ""
RECOVERY_EXPECTED_JOB_RESPONSE_JSON = ""


In [ ]:
from datetime import datetime, timezone
from decimal import Decimal
import hashlib
import json
from pathlib import Path
import uuid

from pyspark.sql import DataFrame, Row, functions as F, types as T

RELEASE_PROFILES = {
    "v1.1.0": {
        "source_root": "Files/landing/synthetic/drill_run/v1.1.0",
        "dataset_version": "v1.1.0",
        "contract_version": "1.1.0",
        "generator_version": "1.1.0",
        "generation_run_id": "13188ed2-899a-5335-ab49-939dbdc961e0",
        "deterministic_seed": 20261023,
        "scenario": "synthetic-proximity-planning-v1",
        "manifest_sha256": "904a5b293fdf0654fb1c5b4d28dd79d1c9cfaaac612f71d8f297ec874f4d284d",
        "source_inventory_sha256": "e50b76b80d6229a7f26bb719a6c937ac83e94762aeb056f3341de8b5014d6dd0",
        "aggregate_source_sha256": "197d65dd19ff59c6ba9ab93fdb1f5cce7020836bec9f6e80550ff07a328b893f",
        "prior_load_evidence_contract": "evidence/fabric-load-v1.1.0.json",
        "expected_file_hashes": {
            "data_dictionary.md": "26379a95847b03f03dfbe5fa345416bac1ddeacad671d1b9dae3bd5b3ea32227",
            "drilling_runs.csv": "bfba06568fd047a7c3cf48a67c1cb6f436f7c85506644684692995d1825da953",
            "generator_config.json": "e4b2174d0e731ce0e171a0d24fb82e33724f89e75ea0d05fd36a11b6fc4cbf41",
            "planned_targets.csv": "b7baa0d1ee87fabed43d8954aa801ab0970ba7be0a1aca1713982bc9350ad711",
            "quality_issues.csv": "03567a8ea3f39807a0ca8361370045aff8a2e9758f0d22c2beffc46929a1b2c2",
            "release_manifest.json": "904a5b293fdf0654fb1c5b4d28dd79d1c9cfaaac612f71d8f297ec874f4d284d",
            "schema.json": "f71c196449c17ab20894448fb8622ec6e96016d74f09dce7885dad90076e1aca",
            "sections.csv": "93346220960a8d0b6475f12dc97ce2d1485a76c877c258bb937a7e709a20515a",
            "source_fingerprint_inventory.json": "e50b76b80d6229a7f26bb719a6c937ac83e94762aeb056f3341de8b5014d6dd0",
            "split_assignments.csv": "13bf5e4073c952c4c23010eff697c5b0cc5db75d779830fc73c7c816127fdec9",
            "wells.csv": "43f84a20f7769fdc97e51ad3273b35171ae12ed29507175578bf374560774ce1",
        },
        "expected_source_rows": {
            "wells.csv": 100,
            "sections.csv": 274,
            "drilling_runs.csv": 937,
            "planned_targets.csv": 12,
            "quality_issues.csv": 20,
            "split_assignments.csv": 100,
        },
        "expected_release_table_rows": {
            "raw.__well_release_data": 100,
            "raw.__section_release_data": 274,
            "raw.__drilling_run_release_data": 937,
            "raw.__run_event_release_data": 0,
            "raw.__quality_issue_release_data": 20,
            "raw.__split_assignment_release_data": 100,
            "raw.__rejected_record_release_data": 0,
            "curated.__well_release_data": 100,
            "curated.__section_release_data": 274,
            "curated.__drilling_run_release_data": 932,
            "curated.__run_event_release_data": 0,
            "curated.__drilling_run_quarantine_release_data": 5,
            "curated.__quality_issue_release_data": 20,
            "curated.__split_assignment_release_data": 100,
            "curated.__rejected_record_release_data": 0,
            "serving.__well_summary_release_data": 100,
            "serving.__dataset_summary_release_data": 1,
            "serving.__quality_summary_release_data": 4,
            "serving.__category_summary_release_data": 19,
            "serving.__offset_run_config_summary_release_data": 0,
            "serving.__offset_run_event_summary_release_data": 0,
            "serving.__planned_target_summary_release_data": 12,
            "serving.__offset_section_summary_release_data": 3288,
        },
        "planned_target_non_null_minimums": {
            "planned_bit_type": 1,
            "planned_bha_type": 1,
            "planned_mud_type": 1,
        },
        "include_run_events": False,
    },
    "v1.2.0": {
        "source_root": "Files/landing/synthetic/drill_run/v1.2.0",
        "dataset_version": "v1.2.0",
        "contract_version": "1.2.0",
        "generator_version": "1.2.0",
        "generation_run_id": "60997560-d118-5dbd-8d76-5e41f69abaf0",
        "deterministic_seed": 20261023,
        "scenario": "synthetic-offset-comparison-context-v1",
        "manifest_sha256": "6370024a3cdec3e3a4fdb3e9525bfe54d88d92375af7a74836461520e5617ab0",
        "source_inventory_sha256": "6d41ceaadd69c35fb558c00c87c3b6c7a9ce3efe614e4ffd16ad4a7787c62ddc",
        "aggregate_source_sha256": "8c1164c27b41b1fe3a359185a4992a2bf02737083b753e1dc3d30d7a8cb12826",
        "prior_load_evidence_contract": "evidence/fabric-load-v1.2.0.json",
        "expected_file_hashes": {
            "data_dictionary.md": "8ff62d6aa007e91714194be80f18c30cb10ab276523ed5f12b91cf06d9b53ad4",
            "drilling_runs.csv": "a67ca6aa09e052e8d7d33ef2939830e72b16eb018a75a88d302b09ebecee2b13",
            "generator_config.json": "53f8e9a8a18c0167f6a613c101af5bf9f73dfac7b779bffce13c99c9d0686742",
            "planned_targets.csv": "70f2865d34b3b7b6691f1e982487632e08b497d9745d40047c33206b47abaa9b",
            "quality_issues.csv": "f76c4bb5c96b21aeb0e0e7419a3dddb148d6fe6b58d70702b89fc26fda5197a1",
            "release_manifest.json": "6370024a3cdec3e3a4fdb3e9525bfe54d88d92375af7a74836461520e5617ab0",
            "run_events.csv": "d428c43bc2506070f046c34f55c6101398cb5cd896a221425c39dde0ae5a5830",
            "schema.json": "4ae1693edf905f143edc2b71f75cfac399eceafe1726ee00298d2321f667d39e",
            "sections.csv": "9129ea53dd37f6a9ecc2e02d3f26a9aea159069e97f5667dede414c744194297",
            "source_fingerprint_inventory.json": "6d41ceaadd69c35fb558c00c87c3b6c7a9ce3efe614e4ffd16ad4a7787c62ddc",
            "split_assignments.csv": "771e68c9d87fb939ab7d3997db97c8cbe4088c407d62fc1514757702d6d72f0a",
            "wells.csv": "3253f8e39a7e3f76f86f3540d3ec2268b7d563138300e293de73328ff0ee699f",
        },
        "expected_source_rows": {
            "wells.csv": 100,
            "sections.csv": 274,
            "drilling_runs.csv": 937,
            "planned_targets.csv": 12,
            "quality_issues.csv": 20,
            "run_events.csv": 927,
            "split_assignments.csv": 100,
        },
        "expected_release_table_rows": {
            "raw.__well_release_data": 100,
            "raw.__section_release_data": 274,
            "raw.__drilling_run_release_data": 937,
            "raw.__run_event_release_data": 927,
            "raw.__quality_issue_release_data": 20,
            "raw.__split_assignment_release_data": 100,
            "raw.__rejected_record_release_data": 0,
            "curated.__well_release_data": 100,
            "curated.__section_release_data": 274,
            "curated.__drilling_run_release_data": 932,
            "curated.__run_event_release_data": 927,
            "curated.__drilling_run_quarantine_release_data": 5,
            "curated.__quality_issue_release_data": 20,
            "curated.__split_assignment_release_data": 100,
            "curated.__rejected_record_release_data": 0,
            "serving.__well_summary_release_data": 100,
            "serving.__dataset_summary_release_data": 1,
            "serving.__quality_summary_release_data": 4,
            "serving.__category_summary_release_data": 19,
            "serving.__offset_run_config_summary_release_data": 11184,
            "serving.__offset_run_event_summary_release_data": 11124,
            "serving.__planned_target_summary_release_data": 12,
            "serving.__offset_section_summary_release_data": 3288,
        },
        "planned_target_non_null_minimums": {
            "planned_bit_type": 1,
            "planned_bha_type": 1,
            "planned_mud_type": 1,
        },
        "include_run_events": True,
    },
}

DEFAULT_RELEASE_VERSION = "v1.2.0"
raw_release_version = globals().get("RELEASE_VERSION", DEFAULT_RELEASE_VERSION)
if not isinstance(raw_release_version, str):
    raise RuntimeError(f"INVALID_RELEASE_VERSION_PARAMETER: {raw_release_version}")
RELEASE_VERSION = raw_release_version.strip()
if RELEASE_VERSION not in RELEASE_PROFILES:
    raise RuntimeError(
        f"UNKNOWN_RELEASE_VERSION: release_version={RELEASE_VERSION} allowed={sorted(RELEASE_PROFILES)}"
    )
RELEASE_PROFILE = RELEASE_PROFILES[RELEASE_VERSION]

for override_name, expected_value in {
    "DATASET_VERSION": RELEASE_PROFILE["dataset_version"],
    "SOURCE_ROOT": RELEASE_PROFILE["source_root"],
    "MANIFEST_SHA256": RELEASE_PROFILE["manifest_sha256"],
    "SOURCE_INVENTORY_SHA256": RELEASE_PROFILE["source_inventory_sha256"],
    "AGGREGATE_SOURCE_SHA256": RELEASE_PROFILE["aggregate_source_sha256"],
}.items():
    override_value = globals().get(override_name)
    if override_value is not None and str(override_value).strip() != str(expected_value):
        raise RuntimeError(
            f"MISMATCHED_RELEASE_INPUT_OVERRIDE: {override_name} expected={expected_value} actual={override_value}"
        )

SOURCE_ROOT = RELEASE_PROFILE["source_root"]
SOURCE_MOUNT_ROOT = f"/lakehouse/default/{SOURCE_ROOT}"
DATASET_VERSION = RELEASE_PROFILE["dataset_version"]
CONTRACT_VERSION = RELEASE_PROFILE["contract_version"]
GENERATOR_VERSION = RELEASE_PROFILE["generator_version"]
GENERATION_RUN_ID = RELEASE_PROFILE["generation_run_id"]
DETERMINISTIC_SEED = RELEASE_PROFILE["deterministic_seed"]
SCENARIO = RELEASE_PROFILE["scenario"]
MANIFEST_SHA256 = RELEASE_PROFILE["manifest_sha256"]
SOURCE_INVENTORY_SHA256 = RELEASE_PROFILE["source_inventory_sha256"]
AGGREGATE_SOURCE_SHA256 = RELEASE_PROFILE["aggregate_source_sha256"]
PRIOR_LOAD_EVIDENCE_CONTRACT = RELEASE_PROFILE["prior_load_evidence_contract"]
FABRIC_WORKSPACE_ID = "4e1977a8-d7ff-487e-b42d-c4b3bc118f3d"
FABRIC_LAKEHOUSE_ID = "964afbf5-6a33-46dc-98c1-be726d0f4999"
FABRIC_LAKEHOUSE_NAME = "lh_drillrun_summary"
FABRIC_NOTEBOOK_NAME = "nb_10_build_drillrun_summary"
PROCESSED_AT_UTC = datetime.now(timezone.utc)

EXPECTED_FILE_HASHES = dict(RELEASE_PROFILE["expected_file_hashes"])
EXPECTED_SOURCE_ROWS = dict(RELEASE_PROFILE["expected_source_rows"])
INCLUDE_RUN_EVENTS = bool(RELEASE_PROFILE["include_run_events"])
EXPECTED_RELEASE_TABLE_ROWS = RELEASE_PROFILE["expected_release_table_rows"]
PLANNED_TARGET_NON_NULL_MINIMUMS = RELEASE_PROFILE["planned_target_non_null_minimums"]
EXPECTED_QUARANTINE_IDS = {
    "SYN-RUN-0013-01-05",
    "SYN-RUN-0021-01-02",
    "SYN-RUN-0064-03-02",
    "SYN-RUN-0069-04-03",
    "SYN-RUN-0071-02-01",
}


def require(condition: bool, code: str, detail: str) -> None:
    if not condition:
        raise RuntimeError(f"{code}: {detail}")


def file_sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


landing_entries = notebookutils.fs.ls(SOURCE_ROOT)
landing_files = sorted(Path(entry.path.rstrip("/")).name for entry in landing_entries if not entry.isDir)
require(
    landing_files == sorted(EXPECTED_FILE_HASHES),
    "SOURCE_INVENTORY_MISMATCH",
    f"expected={sorted(EXPECTED_FILE_HASHES)} actual={landing_files}",
)

actual_file_hashes = {
    filename: file_sha256(f"{SOURCE_MOUNT_ROOT}/{filename}")
    for filename in sorted(EXPECTED_FILE_HASHES)
}
require(
    actual_file_hashes == EXPECTED_FILE_HASHES,
    "SOURCE_HASH_MISMATCH",
    json.dumps(
        {
            name: {"expected": EXPECTED_FILE_HASHES[name], "actual": actual_file_hashes.get(name)}
            for name in sorted(EXPECTED_FILE_HASHES)
            if actual_file_hashes.get(name) != EXPECTED_FILE_HASHES[name]
        },
        sort_keys=True,
    ),
)

with open(f"{SOURCE_MOUNT_ROOT}/release_manifest.json", encoding="utf-8") as stream:
    release_manifest = json.load(stream)
with open(f"{SOURCE_MOUNT_ROOT}/source_fingerprint_inventory.json", encoding="utf-8") as stream:
    source_inventory = json.load(stream)

require(release_manifest["dataset_version"] == DATASET_VERSION, "MANIFEST_VERSION_MISMATCH", str(release_manifest["dataset_version"]))
require(release_manifest["generator_version"] == GENERATOR_VERSION, "MANIFEST_GENERATOR_MISMATCH", str(release_manifest["generator_version"]))
require(release_manifest["generation_run_id"] == GENERATION_RUN_ID, "MANIFEST_RUN_ID_MISMATCH", str(release_manifest["generation_run_id"]))
require(release_manifest["deterministic_seed"] == DETERMINISTIC_SEED, "MANIFEST_SEED_MISMATCH", str(release_manifest["deterministic_seed"]))
require(release_manifest["scenario"] == SCENARIO, "MANIFEST_SCENARIO_MISMATCH", str(release_manifest["scenario"]))
require(release_manifest["is_synthetic"] is True, "MANIFEST_SYNTHETIC_FLAG_INVALID", str(release_manifest["is_synthetic"]))
require(release_manifest["row_counts"] == EXPECTED_SOURCE_ROWS, "MANIFEST_ROW_COUNTS_MISMATCH", json.dumps(release_manifest["row_counts"], sort_keys=True))
require(
    release_manifest["sha256"] == {name: digest for name, digest in EXPECTED_FILE_HASHES.items() if name != "release_manifest.json"},
    "MANIFEST_ARTIFACT_HASHES_MISMATCH",
    "manifest artifact hash map differs from the authorized release",
)
require(
    release_manifest["source_fingerprint"]["value"] == AGGREGATE_SOURCE_SHA256,
    "AGGREGATE_SOURCE_HASH_MISMATCH",
    str(release_manifest["source_fingerprint"]["value"]),
)
require(
    source_inventory["aggregate_source_fingerprint"] == release_manifest["source_fingerprint"],
    "SOURCE_INVENTORY_CONTENT_MISMATCH",
    "aggregate source fingerprint differs between release artifacts",
)

print(json.dumps({"stage": "source_verification", "status": "PASS", "source_path": SOURCE_ROOT, "file_hashes": actual_file_hashes}, sort_keys=True))



In [ ]:
STRING = T.StringType()
INTEGER = T.IntegerType()
LONG = T.LongType()
DECIMAL = T.DecimalType(18, 2)
DECIMAL3 = T.DecimalType(18, 3)
DATE = T.DateType()
TIMESTAMP = T.TimestampType()
BOOLEAN = T.BooleanType()

WELL_FIELDS = [
    ("well_id", STRING, False), ("well_name", STRING, False), ("operator_name", STRING, False),
    ("basin_name", STRING, False), ("field_name", STRING, False), ("spud_date", DATE, False),
    ("completion_date", DATE, False), ("total_footage_ft", DECIMAL, False),
    ("total_duration_hours", DECIMAL, False), ("section_count", INTEGER, False),
    ("run_count", INTEGER, False), ("split_assignment", STRING, False),
    ("coordinate_frame_id", STRING, False), ("location_semantics", STRING, False),
    ("surface_x_ft", INTEGER, False), ("surface_y_ft", INTEGER, False),
    ("is_synthetic", BOOLEAN, False), ("dataset_version", STRING, False),
    ("generator_version", STRING, False), ("generation_run_id", STRING, False),
    ("deterministic_seed", INTEGER, False), ("generated_at_utc", TIMESTAMP, False),
    ("scenario", STRING, False), ("quality_status", STRING, False),
    ("quality_reason_codes", STRING, False),
]
SECTION_FIELDS = [
    ("section_id", STRING, False), ("well_id", STRING, False), ("section_sequence", INTEGER, False),
    ("hole_size_in", DECIMAL, False), ("start_md_ft", DECIMAL, False), ("end_md_ft", DECIMAL, False),
    ("footage_ft", DECIMAL, False), ("duration_hours", DECIMAL, False),
    ("formation_name", STRING, True), ("formation_name_raw", STRING, True),
    ("trajectory_profile", STRING, False), ("run_count", INTEGER, False),
    ("is_synthetic", BOOLEAN, False), ("dataset_version", STRING, False),
    ("generator_version", STRING, False), ("generation_run_id", STRING, False),
    ("deterministic_seed", INTEGER, False), ("generated_at_utc", TIMESTAMP, False),
    ("scenario", STRING, False), ("quality_status", STRING, False),
    ("quality_reason_codes", STRING, False),
]
RUN_FIELDS = [
    ("run_id", STRING, False), ("well_id", STRING, False), ("section_id", STRING, False),
    ("run_sequence_in_section", INTEGER, False), ("start_time_utc", TIMESTAMP, False),
    ("end_time_utc", TIMESTAMP, False), ("start_md_ft", DECIMAL, False),
    ("end_md_ft", DECIMAL, False), ("footage_ft", DECIMAL, False),
    ("duration_hours", DECIMAL, False), ("formation_name", STRING, True),
    ("formation_name_raw", STRING, True), ("bit_type", STRING, False),
    ("bit_size_in", DECIMAL, False), ("bit_iadc_code", STRING, True),
    ("bha_type", STRING, False), ("mud_type", STRING, False),
    ("mud_weight_ppg", DECIMAL, False), ("mud_funnel_viscosity_sec", DECIMAL, False),
    ("is_synthetic", BOOLEAN, False), ("dataset_version", STRING, False),
    ("generator_version", STRING, False), ("generation_run_id", STRING, False),
    ("deterministic_seed", INTEGER, False), ("generated_at_utc", TIMESTAMP, False),
    ("scenario", STRING, False), ("quality_status", STRING, False),
    ("quality_reason_codes", STRING, False),
]
ISSUE_FIELDS = [
    ("quality_issue_id", STRING, False), ("entity_type", STRING, False), ("entity_id", STRING, False),
    ("well_id", STRING, False), ("section_id", STRING, True), ("run_id", STRING, True),
    ("reason_code", STRING, False), ("severity", STRING, False), ("field_name", STRING, False),
    ("observed_value", STRING, True), ("expected_constraint", STRING, False),
    ("handling", STRING, False), ("is_synthetic", BOOLEAN, False),
    ("dataset_version", STRING, False), ("generator_version", STRING, False),
    ("generation_run_id", STRING, False), ("deterministic_seed", INTEGER, False),
    ("generated_at_utc", TIMESTAMP, False), ("scenario", STRING, False),
    ("quality_status", STRING, False), ("quality_reason_codes", STRING, False),
]
SPLIT_FIELDS = [
    ("well_id", STRING, False), ("split_assignment", STRING, False),
    ("assignment_basis", STRING, False), ("is_synthetic", BOOLEAN, False),
    ("dataset_version", STRING, False), ("generator_version", STRING, False),
    ("generation_run_id", STRING, False), ("deterministic_seed", INTEGER, False),
    ("generated_at_utc", TIMESTAMP, False), ("scenario", STRING, False),
    ("quality_status", STRING, False), ("quality_reason_codes", STRING, False),
]
TARGET_FIELDS = [
    ("target_id", STRING, False), ("record_kind", STRING, False), ("basin_name", STRING, False),
    ("field_name", STRING, False), ("expected_formation_name", STRING, False),
    ("planned_trajectory_profile", STRING, False), ("planned_hole_size_in", DECIMAL3, False),
    ("planned_md_min_ft", INTEGER, False), ("planned_md_max_ft", INTEGER, False),
    ("planned_bit_type", STRING, True), ("planned_bha_type", STRING, True),
    ("planned_mud_type", STRING, True),
    ("coordinate_frame_id", STRING, False), ("location_semantics", STRING, False),
    ("surface_x_ft", INTEGER, False), ("surface_y_ft", INTEGER, False),
    ("reference_dataset_version", STRING, False), ("reference_manifest_sha256", STRING, False),
    ("is_synthetic", BOOLEAN, False), ("dataset_version", STRING, False),
    ("generator_version", STRING, False), ("generation_run_id", STRING, False),
    ("deterministic_seed", INTEGER, False), ("generated_at_utc", TIMESTAMP, False),
    ("scenario", STRING, False),
]
RUN_EVENT_FIELDS = [
    ("event_id", STRING, False), ("well_id", STRING, False), ("section_id", STRING, False),
    ("run_id", STRING, False), ("event_type", STRING, False), ("event_subtype", STRING, False),
    ("event_count", INTEGER, False), ("event_duration_hours", DECIMAL, False),
    ("is_synthetic", BOOLEAN, False), ("dataset_version", STRING, False),
    ("generator_version", STRING, False), ("generation_run_id", STRING, False),
    ("deterministic_seed", INTEGER, False), ("generated_at_utc", TIMESTAMP, False),
    ("scenario", STRING, False), ("quality_status", STRING, False),
    ("quality_reason_codes", STRING, False),
]

SOURCE_DEFINITIONS = {
    "wells.csv": WELL_FIELDS,
    "sections.csv": SECTION_FIELDS,
    "drilling_runs.csv": RUN_FIELDS,
    "planned_targets.csv": TARGET_FIELDS,
    "quality_issues.csv": ISSUE_FIELDS,
    "split_assignments.csv": SPLIT_FIELDS,
}
if INCLUDE_RUN_EVENTS:
    SOURCE_DEFINITIONS["run_events.csv"] = RUN_EVENT_FIELDS
PROVENANCE_COLUMNS = [
    "_source_file", "_source_file_sha256", "_source_row_number", "_source_path",
    "_record_hash", "_transformation_contract_version", "_processed_at_utc",
]


def typed_value(column: F.Column, data_type: T.DataType, nullable: bool) -> F.Column:
    if isinstance(data_type, T.StringType):
        value = column
    elif isinstance(data_type, T.BooleanType):
        value = (
            F.when(F.lower(column) == F.lit("true"), F.lit(True))
            .when(F.lower(column) == F.lit("false"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean"))
        )
    elif isinstance(data_type, T.DateType):
        value = F.to_date(column, "yyyy-MM-dd")
    elif isinstance(data_type, T.TimestampType):
        value = F.to_timestamp(column, "yyyy-MM-dd'T'HH:mm:ssX")
    else:
        value = column.cast(data_type)
    if nullable:
        return F.when((column.isNull()) | (column == F.lit("")), F.lit(None).cast(data_type)).otherwise(value)
    return value


def record_hash(fields: list[tuple[str, T.DataType, bool]]) -> F.Column:
    encoded = []
    for name, _, _ in fields:
        original = F.coalesce(F.col(f"_parsed.`{name}`"), F.lit(""))
        encoded.append(F.concat(F.length(F.encode(original, "UTF-8")).cast("string"), F.lit(":"), original))
    return F.sha2(F.concat(*encoded), 256)


def parsed_value(name: str) -> F.Column:
    value = F.col(f"_parsed.`{name}`")
    if name == "quality_reason_codes":
        return F.coalesce(value, F.lit(""))
    return value


REJECTED_SCHEMA = T.StructType([
    T.StructField("dataset_version", STRING, False),
    T.StructField("source_file", STRING, False),
    T.StructField("source_row_number", LONG, False),
    T.StructField("raw_payload", STRING, False),
    T.StructField("rejection_stage", STRING, False),
    T.StructField("rejection_reason_code", STRING, False),
    T.StructField("rejection_detail", STRING, False),
    T.StructField("_source_file_sha256", STRING, False),
    T.StructField("_source_path", STRING, False),
    T.StructField("_record_hash", STRING, False),
    T.StructField("_transformation_contract_version", STRING, False),
    T.StructField("_processed_at_utc", TIMESTAMP, False),
])


def read_typed_source(filename: str, fields: list[tuple[str, T.DataType, bool]]) -> tuple[DataFrame, DataFrame]:
    source_path = f"{SOURCE_ROOT}/{filename}"
    physical_path = f"{SOURCE_MOUNT_ROOT}/{filename}"
    line_schema = T.StructType([
        T.StructField("raw_payload", STRING, False),
        T.StructField("_source_row_number", LONG, False),
    ])
    lines = spark.createDataFrame(
        spark.sparkContext.textFile(source_path)
        .zipWithIndex()
        .filter(lambda pair: pair[1] > 0)
        .map(lambda pair: (pair[0], pair[1])),
        line_schema,
    )
    parse_schema = T.StructType(
        [T.StructField(name, STRING, True) for name, _, _ in fields]
        + [T.StructField("_corrupt_record", STRING, True)]
    )
    parsed = lines.withColumn(
        "_parsed",
        F.from_csv(
            "raw_payload",
            parse_schema.simpleString(),
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record",
                "nullValue": "__FABRIC_CONTRACT_NULL_SENTINEL__",
                "emptyValue": "",
            },
        ),
    )
    invalid_checks = [F.col("_parsed._corrupt_record").isNotNull()]
    for name, data_type, nullable in fields:
        original = parsed_value(name)
        converted = typed_value(original, data_type, nullable)
        if nullable:
            invalid_checks.append((original.isNotNull()) & (original != F.lit("")) & converted.isNull())
        else:
            invalid_checks.append((original.isNull()) | ((original == F.lit("")) & (name != "quality_reason_codes")) | converted.isNull())
    invalid = invalid_checks[0]
    for check in invalid_checks[1:]:
        invalid = invalid | check

    rejected = (
        parsed.filter(invalid)
        .select(
            F.lit(DATASET_VERSION).alias("dataset_version"),
            F.lit(filename).alias("source_file"),
            F.col("_source_row_number").alias("source_row_number"),
            "raw_payload",
            F.lit("RAW_PARSE").alias("rejection_stage"),
            F.lit("TYPE_PARSE_FAILED").alias("rejection_reason_code"),
            F.lit("CSV row failed explicit schema parsing").alias("rejection_detail"),
            F.lit(EXPECTED_FILE_HASHES[filename]).alias("_source_file_sha256"),
            F.lit(source_path).alias("_source_path"),
            F.sha2("raw_payload", 256).alias("_record_hash"),
            F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
            F.lit(PROCESSED_AT_UTC).cast("timestamp").alias("_processed_at_utc"),
        )
    )
    typed_columns = [
        typed_value(parsed_value(name), data_type, nullable).alias(name)
        for name, data_type, nullable in fields
    ]
    valid = (
        parsed.filter(~invalid)
        .select(
            *typed_columns,
            F.lit(filename).alias("_source_file"),
            F.lit(EXPECTED_FILE_HASHES[filename]).alias("_source_file_sha256"),
            "_source_row_number",
            F.lit(source_path).alias("_source_path"),
            record_hash(fields).alias("_record_hash"),
            F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
            F.lit(PROCESSED_AT_UTC).cast("timestamp").alias("_processed_at_utc"),
        )
    )
    return valid, rejected


raw_frames = {}
rejected_frames = []
for source_file, field_definition in SOURCE_DEFINITIONS.items():
    raw_frames[source_file], rejected = read_typed_source(source_file, field_definition)
    raw_frames[source_file].cache()
    rejected_frames.append(rejected)

raw_rejected = spark.createDataFrame([], REJECTED_SCHEMA)
for rejected in rejected_frames:
    raw_rejected = raw_rejected.unionByName(rejected)
raw_rejected.cache()

raw_well = raw_frames["wells.csv"]
raw_section = raw_frames["sections.csv"]
raw_run = raw_frames["drilling_runs.csv"]
raw_issue = raw_frames["quality_issues.csv"]
raw_split = raw_frames["split_assignments.csv"]
raw_target = raw_frames["planned_targets.csv"]
raw_run_event = raw_frames.get("run_events.csv")
raw_event = raw_run_event
if raw_event is None:
    raw_event = spark.createDataFrame([], T.StructType([T.StructField(name, data_type, nullable) for name, data_type, nullable in RUN_EVENT_FIELDS] + [
        T.StructField("_source_file", STRING, False),
        T.StructField("_source_file_sha256", STRING, False),
        T.StructField("_source_row_number", LONG, False),
        T.StructField("_source_path", STRING, False),
        T.StructField("_record_hash", STRING, False),
        T.StructField("_transformation_contract_version", STRING, False),
        T.StructField("_processed_at_utc", TIMESTAMP, False),
    ]))

actual_source_rows = {
    filename: raw_frames[filename].count() + rejected_frames[index].count()
    for index, filename in enumerate(SOURCE_DEFINITIONS)
}
require(actual_source_rows == EXPECTED_SOURCE_ROWS, "SOURCE_ROW_COUNT_MISMATCH", json.dumps(actual_source_rows, sort_keys=True))

if raw_rejected.count() != 0:
    display(raw_rejected.orderBy("source_file", "source_row_number"))
    raise RuntimeError("RAW_REJECTED_RECORDS_PRESENT: release staging blocked before any Delta data write")



In [ ]:
def require_count(frame: DataFrame, expected: int, code: str) -> None:
    actual = frame.count()
    require(actual == expected, code, f"expected={expected} actual={actual}")


def require_unique(frame: DataFrame, keys: list[str], code: str) -> None:
    null_count = frame.filter(F.expr(" OR ".join(f"`{key}` IS NULL" for key in keys))).count()
    duplicate_count = frame.groupBy(*keys).count().filter(F.col("count") != 1).count()
    require(null_count == 0 and duplicate_count == 0, code, f"null_keys={null_count} duplicate_keys={duplicate_count}")


def require_release_constants(frame: DataFrame, name: str) -> None:
    invalid = frame.filter(
        (F.col("dataset_version") != DATASET_VERSION)
        | (F.col("generator_version") != GENERATOR_VERSION)
        | (F.col("generation_run_id") != GENERATION_RUN_ID)
        | (F.col("deterministic_seed") != DETERMINISTIC_SEED)
        | (F.col("scenario") != SCENARIO)
        | (F.col("is_synthetic") != F.lit(True))
    ).count()
    require(invalid == 0, "RELEASE_CONSTANT_MISMATCH", f"{name} invalid_rows={invalid}")


raw_validations = [
    (raw_well, 100, "well", ["dataset_version", "well_id"]),
    (raw_section, 274, "section", ["dataset_version", "section_id"]),
    (raw_run, 937, "drilling_run", ["dataset_version", "run_id"]),
    (raw_issue, 20, "quality_issue", ["dataset_version", "quality_issue_id"]),
    (raw_split, 100, "split_assignment", ["dataset_version", "well_id"]),
    (raw_target, 12, "planned_target", ["dataset_version", "target_id"]),
]
if INCLUDE_RUN_EVENTS:
    require(raw_run_event is not None, "RUN_EVENTS_REQUIRED_BY_RELEASE", DATASET_VERSION)
    raw_validations.append(
        (raw_run_event, EXPECTED_SOURCE_ROWS["run_events.csv"], "run_event", ["dataset_version", "event_id"])
    )
else:
    require(raw_run_event is None, "RUN_EVENTS_UNEXPECTED_FOR_RELEASE", DATASET_VERSION)

for frame, expected, name, keys in raw_validations:
    require_count(frame, expected, f"RAW_{name.upper()}_COUNT_MISMATCH")
    require_unique(frame, keys, f"RAW_{name.upper()}_KEY_VIOLATION")
    require_release_constants(frame, name)

require(raw_section.join(raw_well.select("dataset_version", "well_id"), ["dataset_version", "well_id"], "left_anti").count() == 0, "SECTION_WELL_RELATIONSHIP_VIOLATION", "orphan section")
require(raw_run.join(raw_well.select("dataset_version", "well_id"), ["dataset_version", "well_id"], "left_anti").count() == 0, "RUN_WELL_RELATIONSHIP_VIOLATION", "orphan run")
require(raw_run.join(raw_section.select("dataset_version", "section_id"), ["dataset_version", "section_id"], "left_anti").count() == 0, "RUN_SECTION_RELATIONSHIP_VIOLATION", "orphan run")
require(raw_split.join(raw_well.select("dataset_version", "well_id"), ["dataset_version", "well_id"], "left_anti").count() == 0, "SPLIT_WELL_RELATIONSHIP_VIOLATION", "orphan split")

issue_well_orphans = raw_issue.join(raw_well.select("dataset_version", "well_id"), ["dataset_version", "well_id"], "left_anti").count()
issue_section_orphans = raw_issue.filter(F.col("entity_type") == "SECTION").join(
    raw_section.select("dataset_version", F.col("section_id").alias("entity_id")),
    ["dataset_version", "entity_id"], "left_anti"
).count()
issue_run_orphans = raw_issue.filter(F.col("entity_type") == "DRILLING_RUN").join(
    raw_run.select("dataset_version", F.col("run_id").alias("entity_id")),
    ["dataset_version", "entity_id"], "left_anti"
).count()
require(issue_well_orphans + issue_section_orphans + issue_run_orphans == 0, "QUALITY_ISSUE_RELATIONSHIP_VIOLATION", f"well={issue_well_orphans} section={issue_section_orphans} run={issue_run_orphans}")

section_sequence_violations = (
    raw_section.groupBy("dataset_version", "well_id")
    .agg(F.min("section_sequence").alias("minimum"), F.max("section_sequence").alias("maximum"), F.count("*").alias("records"))
    .filter((F.col("minimum") != 1) | (F.col("maximum") != F.col("records")))
    .count()
)
run_sequence_violations = (
    raw_run.groupBy("dataset_version", "section_id")
    .agg(F.min("run_sequence_in_section").alias("minimum"), F.max("run_sequence_in_section").alias("maximum"), F.count("*").alias("records"))
    .filter((F.col("minimum") != 1) | (F.col("maximum") != F.col("records")))
    .count()
)
require(section_sequence_violations == 0, "SECTION_SEQUENCE_VIOLATION", str(section_sequence_violations))
require(run_sequence_violations == 0, "RUN_SEQUENCE_VIOLATION", str(run_sequence_violations))

curated_well = raw_well
curated_section = raw_section
curated_split = raw_split
curated_target = raw_target
curated_run = raw_run.filter(F.col("quality_status").isin("PASS", "WARNING"))
curated_event = raw_event.filter(F.col("quality_status").isin("PASS", "WARNING"))
quarantine_issue_key = raw_issue.filter(F.col("reason_code") == "NON_POSITIVE_DURATION").select(
    "dataset_version", "run_id", "reason_code", "quality_issue_id"
)
curated_run_quarantine = (
    raw_run.filter(F.col("quality_status") == "ERROR")
    .withColumn("quarantine_reason_code", F.col("quality_reason_codes"))
    .join(
        quarantine_issue_key,
        (raw_run.dataset_version == quarantine_issue_key.dataset_version)
        & (raw_run.run_id == quarantine_issue_key.run_id)
        & (raw_run.quality_reason_codes == quarantine_issue_key.reason_code),
        "left",
    )
    .drop(quarantine_issue_key.dataset_version)
    .drop(quarantine_issue_key.run_id)
    .drop("reason_code")
)
curated_issue = raw_issue
curated_rejected = raw_rejected.withColumn("curation_disposition", F.lit("BLOCKED"))

require_count(curated_run, 932, "ACCEPTED_RUN_COUNT_MISMATCH")
require_count(curated_run_quarantine, 5, "QUARANTINED_RUN_COUNT_MISMATCH")
require_count(curated_rejected, 0, "REJECTED_RECORD_COUNT_MISMATCH")
require(
    {row.run_id for row in curated_run_quarantine.select("run_id").collect()} == EXPECTED_QUARANTINE_IDS,
    "QUARANTINE_KEY_MISMATCH",
    "quarantined run IDs differ from the contract",
)
require(
    curated_run_quarantine.filter(
        (F.col("duration_hours") > F.lit(0)) | (F.col("quarantine_reason_code") != "NON_POSITIVE_DURATION") | F.col("quality_issue_id").isNull()
    ).count() == 0,
    "QUARANTINE_REASON_VIOLATION",
    "quarantine rows must be linked non-positive-duration errors",
)
require(raw_run.count() == curated_run.count() + curated_run_quarantine.count() + curated_rejected.count(), "RUN_ROUTING_RECONCILIATION_FAILED", "937 must equal 932 + 5 + 0")

expected_issue_counts = {
    ("MISSING_FORMATION", "WARNING"): 5,
    ("INCONSISTENT_FORMATION_LABEL", "WARNING"): 5,
    ("MISSING_BIT_IADC_CODE", "WARNING"): 5,
    ("NON_POSITIVE_DURATION", "ERROR"): 5,
}
actual_issue_counts = {
    (row.reason_code, row.severity): row["count"]
    for row in curated_issue.groupBy("reason_code", "severity").count().collect()
}
require(actual_issue_counts == expected_issue_counts, "QUALITY_REASON_COUNT_MISMATCH", json.dumps({f"{key[0]}|{key[1]}": value for key, value in actual_issue_counts.items()}, sort_keys=True))

well_split_violations = (
    curated_well.alias("w").join(curated_split.alias("s"), ["dataset_version", "well_id"])
    .filter(F.col("w.split_assignment") != F.col("s.split_assignment")).count()
)
require(well_split_violations == 0, "WELL_SPLIT_MISMATCH", str(well_split_violations))

section_rollup = raw_run.groupBy("dataset_version", "section_id").agg(
    F.count("*").alias("actual_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("actual_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("actual_duration_hours"),
)
section_measure_violations = (
    raw_section.join(section_rollup, ["dataset_version", "section_id"])
    .filter(
        (F.col("run_count") != F.col("actual_run_count"))
        | (F.col("footage_ft").cast("decimal(20,2)") != F.col("actual_footage_ft"))
        | (F.col("duration_hours").cast("decimal(20,2)") != F.col("actual_duration_hours"))
    ).count()
)
require(section_measure_violations == 0, "SECTION_MEASURE_RECONCILIATION_FAILED", str(section_measure_violations))

well_rollup = raw_run.groupBy("dataset_version", "well_id").agg(
    F.count("*").alias("actual_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("actual_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("actual_duration_hours"),
)
well_sections = raw_section.groupBy("dataset_version", "well_id").count().withColumnRenamed("count", "actual_section_count")
well_measure_violations = (
    raw_well.join(well_rollup, ["dataset_version", "well_id"]).join(well_sections, ["dataset_version", "well_id"])
    .filter(
        (F.col("section_count") != F.col("actual_section_count"))
        | (F.col("run_count") != F.col("actual_run_count"))
        | (F.col("total_footage_ft").cast("decimal(20,2)") != F.col("actual_footage_ft"))
        | (F.col("total_duration_hours").cast("decimal(20,2)") != F.col("actual_duration_hours"))
    ).count()
)
require(well_measure_violations == 0, "WELL_MEASURE_RECONCILIATION_FAILED", str(well_measure_violations))



In [ ]:
section_by_well = curated_section.groupBy("dataset_version", "well_id").agg(
    F.count("*").cast("int").alias("curated_section_count"),
    F.sum(F.when(F.col("quality_status") == "WARNING", 1).otherwise(0)).cast("int").alias("warning_section_count"),
)
source_run_by_well = raw_run.groupBy("dataset_version", "well_id").agg(
    F.count("*").cast("int").alias("source_run_count"),
    F.min("start_time_utc").alias("first_run_start_time_utc"),
    F.max("end_time_utc").alias("last_run_end_time_utc"),
)
accepted_by_well = curated_run.groupBy("dataset_version", "well_id").agg(
    F.count("*").cast("int").alias("accepted_run_count"),
    F.sum(F.when(F.col("quality_status") == "WARNING", 1).otherwise(0)).cast("int").alias("warning_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("accepted_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("accepted_duration_hours"),
)
quarantine_by_well = curated_run_quarantine.groupBy("dataset_version", "well_id").agg(
    F.count("*").cast("int").alias("quarantined_run_count"),
    F.sum(F.when(F.col("quality_status") == "ERROR", 1).otherwise(0)).cast("int").alias("error_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("quarantined_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("quarantined_duration_hours"),
)
issues_by_well = curated_issue.groupBy("dataset_version", "well_id").count().withColumnRenamed("count", "issue_count")

well_summary = (
    curated_well.alias("w")
    .join(section_by_well, ["dataset_version", "well_id"])
    .join(source_run_by_well, ["dataset_version", "well_id"])
    .join(accepted_by_well, ["dataset_version", "well_id"])
    .join(quarantine_by_well, ["dataset_version", "well_id"], "left")
    .join(issues_by_well, ["dataset_version", "well_id"], "left")
    .fillna(0, subset=[
        "quarantined_run_count", "error_run_count", "issue_count",
        "quarantined_footage_ft", "quarantined_duration_hours",
    ])
    .select(
        "dataset_version", "w.generator_version", "w.generation_run_id", "w.scenario",
        "w.deterministic_seed", "w.is_synthetic", "well_id", "w.well_name",
        "w.operator_name", "w.basin_name", "w.field_name", "w.spud_date",
        "w.completion_date", "w.split_assignment", F.col("curated_section_count").alias("section_count"), "source_run_count",
        "accepted_run_count", F.col("quarantined_run_count").cast("int"),
        "warning_section_count", "warning_run_count", F.col("error_run_count").cast("int"),
        F.col("issue_count").cast("int"),
        F.col("w.total_footage_ft").cast("decimal(20,2)").alias("source_total_footage_ft"),
        "accepted_footage_ft", F.col("quarantined_footage_ft").cast("decimal(20,2)"),
        F.col("w.total_duration_hours").cast("decimal(20,2)").alias("source_total_duration_hours"),
        "accepted_duration_hours", F.col("quarantined_duration_hours").cast("decimal(20,2)"),
        "first_run_start_time_utc", "last_run_end_time_utc",
        ((F.col("warning_section_count") + F.col("warning_run_count")) > 0).alias("has_quality_warning"),
        (F.col("quarantined_run_count") > 0).alias("has_quarantined_run"),
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)

source_totals = raw_run.agg(
    F.count("*").cast("int").alias("source_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("source_total_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("source_total_duration_hours"),
)
accepted_totals = curated_run.agg(
    F.count("*").cast("int").alias("accepted_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("accepted_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("accepted_duration_hours"),
)
quarantine_totals = curated_run_quarantine.agg(
    F.count("*").cast("int").alias("quarantined_run_count"),
    F.sum("footage_ft").cast("decimal(20,2)").alias("quarantined_footage_ft"),
    F.sum("duration_hours").cast("decimal(20,2)").alias("quarantined_duration_hours"),
)
issue_totals = curated_issue.agg(
    F.count("*").cast("int").alias("quality_issue_count"),
    F.sum(F.when(F.col("severity") == "WARNING", 1).otherwise(0)).cast("int").alias("warning_issue_count"),
    F.sum(F.when(F.col("severity") == "ERROR", 1).otherwise(0)).cast("int").alias("error_issue_count"),
)
dataset_summary = (
    source_totals.crossJoin(accepted_totals).crossJoin(quarantine_totals).crossJoin(issue_totals)
    .select(
        F.lit(DATASET_VERSION).alias("dataset_version"),
        F.lit(GENERATOR_VERSION).alias("generator_version"),
        F.lit(GENERATION_RUN_ID).alias("generation_run_id"),
        F.lit(SCENARIO).alias("scenario"),
        F.lit(DETERMINISTIC_SEED).cast("int").alias("deterministic_seed"),
        F.lit(True).alias("is_synthetic"),
        F.lit(100).cast("int").alias("well_count"),
        F.lit(274).cast("int").alias("section_count"),
        "source_run_count", "accepted_run_count", "quarantined_run_count",
        F.lit(0).cast("int").alias("rejected_record_count"),
        "quality_issue_count", "warning_issue_count", "error_issue_count",
        "source_total_footage_ft", "accepted_footage_ft", "quarantined_footage_ft",
        "source_total_duration_hours", "accepted_duration_hours", "quarantined_duration_hours",
        F.lit(MANIFEST_SHA256).alias("manifest_sha256"),
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)

quality_summary = (
    curated_issue.groupBy("dataset_version", "generation_run_id", "scenario", "is_synthetic", "reason_code", "severity")
    .agg(
        F.count("*").cast("int").alias("issue_count"),
        F.countDistinct("well_id").cast("int").alias("affected_well_count"),
        F.countDistinct("section_id").cast("int").alias("affected_section_count"),
        F.countDistinct("run_id").cast("int").alias("affected_run_count"),
        F.first("handling").alias("handling_text"),
        F.countDistinct("handling").alias("_handling_count"),
    )
)
require(quality_summary.filter(F.col("_handling_count") != 1).count() == 0, "QUALITY_HANDLING_CONFLICT", "reason/severity group has multiple handling values")
quality_summary = quality_summary.drop("_handling_count").withColumn("_transformation_contract_version", F.lit(CONTRACT_VERSION))


def category(frame: DataFrame, name: str, value: F.Column) -> DataFrame:
    return (
        frame.withColumn("category_value", value)
        .groupBy("dataset_version", "generation_run_id", "scenario", "is_synthetic", "category_value")
        .count()
        .select(
            "dataset_version", "generation_run_id", "scenario", "is_synthetic",
            F.lit(name).alias("category_name"), "category_value",
            F.col("count").cast("int").alias("record_count"),
            F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
        )
    )


entity_quality = raw_well.select("dataset_version", "generation_run_id", "scenario", "is_synthetic", "quality_status").unionByName(
    raw_section.select("dataset_version", "generation_run_id", "scenario", "is_synthetic", "quality_status")
).unionByName(
    raw_run.select("dataset_version", "generation_run_id", "scenario", "is_synthetic", "quality_status")
)
category_summary = (
    category(raw_section, "FORMATION_NAME", F.coalesce(F.col("formation_name"), F.lit("(missing)")))
    .unionByName(category(raw_run, "BHA_TYPE", F.col("bha_type")))
    .unionByName(category(raw_run, "BIT_TYPE", F.col("bit_type")))
    .unionByName(category(curated_well, "SPLIT_ASSIGNMENT", F.col("split_assignment")))
    .unionByName(category(entity_quality, "ENTITY_QUALITY_STATUS", F.col("quality_status")))
)



planned_target_summary = (
    curated_target.select(
        "dataset_version", "generator_version", "generation_run_id", "scenario",
        "deterministic_seed", "is_synthetic", "target_id", "record_kind",
        F.col("field_name").alias("target_name"),
        F.col("basin_name").alias("target_basin_name"),
        F.col("field_name").alias("target_field_name"),
        "expected_formation_name", "planned_trajectory_profile", "planned_hole_size_in",
        "planned_md_min_ft", "planned_md_max_ft",
        "planned_bit_type", "planned_bha_type", "planned_mud_type",
        "coordinate_frame_id", "location_semantics",
        F.col("surface_x_ft").alias("target_surface_x_ft"),
        F.col("surface_y_ft").alias("target_surface_y_ft"),
        "reference_dataset_version", "reference_manifest_sha256",
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)

section_run_metrics = raw_run.groupBy("dataset_version", "section_id").agg(
    F.count("*").cast("int").alias("source_run_count"),
    F.sum(F.when(F.col("quality_status").isin("PASS", "WARNING"), 1).otherwise(0)).cast("int").alias("accepted_run_count"),
    F.sum(F.when(F.col("quality_status") == "ERROR", 1).otherwise(0)).cast("int").alias("quarantined_run_count"),
    F.sum(F.when(F.col("quality_status") == "WARNING", 1).otherwise(0)).cast("int").alias("warning_run_count"),
    F.sum(F.when(F.col("quality_status") == "ERROR", 1).otherwise(0)).cast("int").alias("error_run_count"),
    F.sum(F.when(F.col("quality_status").isin("PASS", "WARNING"), F.col("footage_ft")).otherwise(F.lit(0))).cast("decimal(20,2)").alias("accepted_footage_ft"),
    F.sum(F.when(F.col("quality_status").isin("PASS", "WARNING"), F.col("duration_hours")).otherwise(F.lit(0))).cast("decimal(20,2)").alias("accepted_duration_hours"),
)

issue_with_section = (
    raw_issue.alias("qi")
    .join(raw_run.select("dataset_version", "run_id", F.col("section_id").alias("run_section_id")), ["dataset_version", "run_id"], "left")
    .withColumn(
        "effective_section_id",
        F.when(F.col("qi.entity_type") == "SECTION", F.col("qi.section_id")).otherwise(F.col("run_section_id")),
    )
)

section_issue_metrics = issue_with_section.groupBy("dataset_version", "effective_section_id").agg(
    F.countDistinct(F.when(F.col("severity") == "WARNING", F.col("quality_issue_id"))).cast("int").alias("warning_issue_count"),
    F.countDistinct(F.when(F.col("severity") == "ERROR", F.col("quality_issue_id"))).cast("int").alias("error_issue_count"),
).withColumnRenamed("effective_section_id", "section_id")

section_metrics = (
    curated_section.alias("s")
    .join(curated_well.alias("w"), ["dataset_version", "well_id"])
    .join(section_run_metrics, ["dataset_version", "section_id"])
    .join(section_issue_metrics, ["dataset_version", "section_id"], "left")
    .fillna(0, subset=["warning_issue_count", "error_issue_count"])
    .select(
        "dataset_version", "well_id", "w.well_name", "w.operator_name",
        F.col("w.basin_name").alias("well_basin_name"),
        F.col("w.field_name").alias("well_field_name"),
        F.col("w.coordinate_frame_id").alias("well_coordinate_frame_id"),
        F.col("w.location_semantics").alias("well_location_semantics"),
        F.col("w.surface_x_ft").alias("well_surface_x_ft"),
        F.col("w.surface_y_ft").alias("well_surface_y_ft"),
        "section_id", "s.section_sequence",
        F.col("s.formation_name").alias("section_formation_name"),
        F.col("s.hole_size_in").cast("decimal(18,3)").alias("section_hole_size_in"),
        F.col("s.start_md_ft").cast("decimal(20,2)").alias("section_start_md_ft"),
        F.col("s.end_md_ft").cast("decimal(20,2)").alias("section_end_md_ft"),
        F.col("s.trajectory_profile").alias("section_trajectory_profile"),
        "source_run_count", "accepted_run_count", "quarantined_run_count",
        "warning_run_count", "error_run_count",
        F.col("warning_issue_count").cast("int"), F.col("error_issue_count").cast("int"),
        "accepted_footage_ft", "accepted_duration_hours",
    )
)

offset_section_summary = (
    curated_target.alias("t")
    .crossJoin(section_metrics.alias("s"))
    .filter(F.col("t.dataset_version") == F.col("s.dataset_version"))
    .filter(F.col("t.coordinate_frame_id") == F.col("s.well_coordinate_frame_id"))
    .filter(F.col("t.location_semantics") == F.col("s.well_location_semantics"))
    .select(
        F.col("t.dataset_version").alias("dataset_version"),
        F.col("t.generator_version").alias("generator_version"),
        F.col("t.generation_run_id").alias("generation_run_id"),
        F.col("t.scenario").alias("scenario"),
        F.col("t.deterministic_seed").alias("deterministic_seed"),
        F.col("t.is_synthetic").alias("is_synthetic"),
        F.col("t.target_id").alias("target_id"),
        F.col("t.field_name").alias("target_name"),
        F.col("t.basin_name").alias("target_basin_name"),
        F.col("t.field_name").alias("target_field_name"),
        F.col("t.expected_formation_name").alias("expected_formation_name"),
        F.col("t.planned_hole_size_in").cast("decimal(18,3)").alias("planned_hole_size_in"),
        F.col("t.planned_md_min_ft").cast("decimal(20,2)").alias("planned_md_min_ft"),
        F.col("t.planned_md_max_ft").cast("decimal(20,2)").alias("planned_md_max_ft"),
        F.col("t.planned_trajectory_profile").alias("planned_trajectory_profile"),
        F.col("t.coordinate_frame_id").alias("coordinate_frame_id"),
        F.col("t.location_semantics").alias("location_semantics"),
        F.col("t.surface_x_ft").alias("target_surface_x_ft"),
        F.col("t.surface_y_ft").alias("target_surface_y_ft"),
        F.col("s.well_id").alias("well_id"),
        F.col("s.well_name").alias("well_name"),
        F.col("s.operator_name").alias("operator_name"),
        F.col("s.well_basin_name").alias("well_basin_name"),
        F.col("s.well_field_name").alias("well_field_name"),
        F.col("s.well_surface_x_ft").alias("well_surface_x_ft"),
        F.col("s.well_surface_y_ft").alias("well_surface_y_ft"),
        F.col("s.section_id").alias("section_id"),
        F.col("s.section_sequence").alias("section_sequence"),
        F.col("s.section_formation_name").alias("section_formation_name"),
        F.col("s.section_hole_size_in").alias("section_hole_size_in"),
        F.col("s.section_start_md_ft").alias("section_start_md_ft"),
        F.col("s.section_end_md_ft").alias("section_end_md_ft"),
        F.col("s.section_trajectory_profile").alias("section_trajectory_profile"),
        F.col("s.source_run_count").alias("source_run_count"),
        F.col("s.accepted_run_count").alias("accepted_run_count"),
        F.col("s.quarantined_run_count").alias("quarantined_run_count"),
        F.col("s.warning_run_count").alias("warning_run_count"),
        F.col("s.error_run_count").alias("error_run_count"),
        F.col("s.warning_issue_count").alias("warning_issue_count"),
        F.col("s.error_issue_count").alias("error_issue_count"),
        F.col("s.accepted_footage_ft").alias("accepted_footage_ft"),
        F.col("s.accepted_duration_hours").alias("accepted_duration_hours"),
        (
            (F.col("t.surface_x_ft").cast("decimal(38,12)") - F.col("s.well_surface_x_ft").cast("decimal(38,12)"))
            * (F.col("t.surface_x_ft").cast("decimal(38,12)") - F.col("s.well_surface_x_ft").cast("decimal(38,12)"))
            +
            (F.col("t.surface_y_ft").cast("decimal(38,12)") - F.col("s.well_surface_y_ft").cast("decimal(38,12)"))
            * (F.col("t.surface_y_ft").cast("decimal(38,12)") - F.col("s.well_surface_y_ft").cast("decimal(38,12)"))
        ).cast("decimal(38,12)").alias("distance_squared_ft2"),
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)


run_config_summary = (
    curated_run.alias("r")
    .join(curated_well.select("dataset_version", "well_id", "well_name"), ["dataset_version", "well_id"], "inner")
    .groupBy(
        "dataset_version", "well_id", "well_name", "bit_type", "bha_type",
        "mud_type", "mud_weight_ppg", "mud_funnel_viscosity_sec",
    )
    .agg(
        F.count("*").cast("int").alias("accepted_run_count"),
        F.sum("footage_ft").cast("decimal(20,2)").alias("accepted_footage_ft"),
        F.sum("duration_hours").cast("decimal(20,2)").alias("accepted_duration_hours"),
    )
)

offset_run_config_summary = (
    curated_target.alias("t")
    .crossJoin(run_config_summary.alias("r"))
    .filter(F.col("t.dataset_version") == F.col("r.dataset_version"))
    .select(
        F.col("t.dataset_version").alias("dataset_version"),
        F.col("t.generator_version").alias("generator_version"),
        F.col("t.generation_run_id").alias("generation_run_id"),
        F.col("t.scenario").alias("scenario"),
        F.col("t.deterministic_seed").alias("deterministic_seed"),
        F.col("t.is_synthetic").alias("is_synthetic"),
        F.col("t.target_id").alias("target_id"),
        F.col("t.field_name").alias("target_name"),
        F.col("r.well_id").alias("well_id"),
        F.col("r.well_name").alias("well_name"),
        F.col("r.bit_type").alias("bit_type"),
        F.col("r.bha_type").alias("bha_type"),
        F.col("r.mud_type").alias("mud_type"),
        F.col("r.mud_weight_ppg").alias("mud_weight_ppg"),
        F.col("r.mud_funnel_viscosity_sec").alias("mud_funnel_viscosity_sec"),
        F.col("r.accepted_run_count").alias("accepted_run_count"),
        F.col("r.accepted_footage_ft").alias("accepted_footage_ft"),
        F.col("r.accepted_duration_hours").alias("accepted_duration_hours"),
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)

offset_run_event_summary = (
    curated_target.alias("t")
    .crossJoin(curated_event.alias("e"))
    .filter(F.col("t.dataset_version") == F.col("e.dataset_version"))
    .join(
        curated_well.select("dataset_version", "well_id", "well_name"),
        ["dataset_version", "well_id"],
        "inner",
    )
    .select(
        F.col("t.dataset_version").alias("dataset_version"),
        F.col("t.generator_version").alias("generator_version"),
        F.col("t.generation_run_id").alias("generation_run_id"),
        F.col("t.scenario").alias("scenario"),
        F.col("t.deterministic_seed").alias("deterministic_seed"),
        F.col("t.is_synthetic").alias("is_synthetic"),
        F.col("t.target_id").alias("target_id"),
        F.col("t.field_name").alias("target_name"),
        F.col("e.well_id").alias("well_id"),
        F.col("well_name").alias("well_name"),
        F.col("e.section_id").alias("section_id"),
        F.col("e.run_id").alias("run_id"),
        F.col("e.event_type").alias("event_type"),
        F.col("e.event_subtype").alias("event_subtype"),
        F.col("e.event_count").alias("event_count"),
        F.col("e.event_duration_hours").alias("event_duration_hours"),
        F.lit(CONTRACT_VERSION).alias("_transformation_contract_version"),
    )
)


In [ ]:
expected_dataset_summary = {
    "well_count": 100, "section_count": 274, "source_run_count": 937,
    "accepted_run_count": 932, "quarantined_run_count": 5, "rejected_record_count": 0,
    "quality_issue_count": 20, "warning_issue_count": 15, "error_issue_count": 5,
    "source_total_footage_ft": Decimal("914297.60"),
    "accepted_footage_ft": Decimal("909010.20"),
    "quarantined_footage_ft": Decimal("5287.40"),
    "source_total_duration_hours": Decimal("22358.76"),
    "accepted_duration_hours": Decimal("22358.76"),
    "quarantined_duration_hours": Decimal("0.00"),
}
actual_dataset_summary = dataset_summary.collect()[0].asDict()
for field_name, expected_value in expected_dataset_summary.items():
    require(actual_dataset_summary[field_name] == expected_value, "DATASET_SUMMARY_RECONCILIATION_FAILED", f"{field_name} expected={expected_value} actual={actual_dataset_summary[field_name]}")

require_count(well_summary, 100, "WELL_SUMMARY_COUNT_MISMATCH")
require_count(dataset_summary, 1, "DATASET_SUMMARY_COUNT_MISMATCH")
require_count(quality_summary, 4, "QUALITY_SUMMARY_COUNT_MISMATCH")
require_count(category_summary, 19, "CATEGORY_SUMMARY_COUNT_MISMATCH")
require_count(planned_target_summary, 12, "PLANNED_TARGET_SUMMARY_COUNT_MISMATCH")
require_count(offset_section_summary, 3288, "OFFSET_SECTION_SUMMARY_COUNT_MISMATCH")
require_count(curated_event, 927, "CURATED_RUN_EVENT_COUNT_MISMATCH")
require_count(offset_run_config_summary, 11184, "OFFSET_RUN_CONFIG_SUMMARY_COUNT_MISMATCH")
require_count(offset_run_event_summary, 11124, "OFFSET_RUN_EVENT_SUMMARY_COUNT_MISMATCH")

expected_categories = {
    ("FORMATION_NAME", "(missing)"): 5,
    ("FORMATION_NAME", "Synthetic Aster"): 27,
    ("FORMATION_NAME", "Synthetic Brimstone"): 28,
    ("FORMATION_NAME", "Synthetic Cobalt"): 76,
    ("FORMATION_NAME", "Synthetic Driftwood"): 56,
    ("FORMATION_NAME", "Synthetic Ember"): 45,
    ("FORMATION_NAME", "Synthetic Flint"): 37,
    ("BHA_TYPE", "MOTOR"): 311, ("BHA_TYPE", "ROTARY"): 301, ("BHA_TYPE", "RSS"): 325,
    ("BIT_TYPE", "HYBRID"): 313, ("BIT_TYPE", "PDC"): 308, ("BIT_TYPE", "TRICONE"): 316,
    ("SPLIT_ASSIGNMENT", "TRAIN"): 70, ("SPLIT_ASSIGNMENT", "VALIDATION"): 15, ("SPLIT_ASSIGNMENT", "TEST"): 15,
    ("ENTITY_QUALITY_STATUS", "PASS"): 1276, ("ENTITY_QUALITY_STATUS", "WARNING"): 30, ("ENTITY_QUALITY_STATUS", "ERROR"): 5,
}
actual_categories = {
    (row.category_name, row.category_value): row.record_count
    for row in category_summary.select("category_name", "category_value", "record_count").collect()
}
require(actual_categories == expected_categories, "CATEGORY_RECONCILIATION_FAILED", json.dumps({f"{key[0]}|{key[1]}": value for key, value in actual_categories.items()}, sort_keys=True))

well_summary_rollup = well_summary.agg(
    F.sum("source_run_count").alias("source_run_count"),
    F.sum("accepted_run_count").alias("accepted_run_count"),
    F.sum("quarantined_run_count").alias("quarantined_run_count"),
    F.sum("source_total_footage_ft").cast("decimal(20,2)").alias("source_total_footage_ft"),
    F.sum("accepted_footage_ft").cast("decimal(20,2)").alias("accepted_footage_ft"),
    F.sum("quarantined_footage_ft").cast("decimal(20,2)").alias("quarantined_footage_ft"),
    F.sum("source_total_duration_hours").cast("decimal(20,2)").alias("source_total_duration_hours"),
    F.sum("accepted_duration_hours").cast("decimal(20,2)").alias("accepted_duration_hours"),
    F.sum("quarantined_duration_hours").cast("decimal(20,2)").alias("quarantined_duration_hours"),
).collect()[0].asDict()
for field_name in well_summary_rollup:
    require(well_summary_rollup[field_name] == actual_dataset_summary[field_name], "WELL_SUMMARY_ROLLUP_MISMATCH", f"{field_name} well={well_summary_rollup[field_name]} dataset={actual_dataset_summary[field_name]}")



In [ ]:
LOGICAL_TABLES = {
    "raw.well": raw_well,
    "raw.section": raw_section,
    "raw.drilling_run": raw_run,
    "raw.run_event": raw_event,
    "raw.quality_issue": raw_issue,
    "raw.split_assignment": raw_split,
    "raw.rejected_record": raw_rejected,
    "curated.well": curated_well,
    "curated.section": curated_section,
    "curated.drilling_run": curated_run,
    "curated.run_event": curated_event,
    "curated.drilling_run_quarantine": curated_run_quarantine,
    "curated.quality_issue": curated_issue,
    "curated.split_assignment": curated_split,
    "curated.rejected_record": curated_rejected,
    "serving.well_summary": well_summary,
    "serving.dataset_summary": dataset_summary,
    "serving.quality_summary": quality_summary,
    "serving.category_summary": category_summary,
    "serving.offset_run_config_summary": offset_run_config_summary,
    "serving.offset_run_event_summary": offset_run_event_summary,
    "serving.planned_target_summary": planned_target_summary,
    "serving.offset_section_summary": offset_section_summary,
}
EXPECTED_TABLE_ROWS = {
    "raw.well": 100, "raw.section": 274, "raw.drilling_run": 937,
    "raw.run_event": EXPECTED_SOURCE_ROWS.get("run_events.csv", 0),
    "raw.quality_issue": 20, "raw.split_assignment": 100, "raw.rejected_record": 0,
    "curated.well": 100, "curated.section": 274, "curated.drilling_run": 932,
    "curated.run_event": EXPECTED_SOURCE_ROWS.get("run_events.csv", 0),
    "curated.drilling_run_quarantine": 5, "curated.quality_issue": 20,
    "curated.split_assignment": 100, "curated.rejected_record": 0,
    "serving.well_summary": 100, "serving.dataset_summary": 1,
    "serving.quality_summary": 4, "serving.category_summary": 19,
    "serving.offset_run_config_summary": EXPECTED_RELEASE_TABLE_ROWS["serving.__offset_run_config_summary_release_data"],
    "serving.offset_run_event_summary": EXPECTED_RELEASE_TABLE_ROWS["serving.__offset_run_event_summary_release_data"],
    "serving.planned_target_summary": 12, "serving.offset_section_summary": 3288,
}
TABLE_KEYS = {
    "raw.well": ["dataset_version", "well_id"],
    "raw.section": ["dataset_version", "section_id"],
    "raw.drilling_run": ["dataset_version", "run_id"],
    "raw.run_event": ["dataset_version", "event_id"],
    "raw.quality_issue": ["dataset_version", "quality_issue_id"],
    "raw.split_assignment": ["dataset_version", "well_id"],
    "raw.rejected_record": ["dataset_version", "source_file", "source_row_number"],
    "curated.well": ["dataset_version", "well_id"],
    "curated.section": ["dataset_version", "section_id"],
    "curated.drilling_run": ["dataset_version", "run_id"],
    "curated.run_event": ["dataset_version", "event_id"],
    "curated.drilling_run_quarantine": ["dataset_version", "run_id"],
    "curated.quality_issue": ["dataset_version", "quality_issue_id"],
    "curated.split_assignment": ["dataset_version", "well_id"],
    "curated.rejected_record": ["dataset_version", "source_file", "source_row_number"],
    "serving.well_summary": ["dataset_version", "well_id"],
    "serving.dataset_summary": ["dataset_version", "generation_run_id"],
    "serving.quality_summary": ["dataset_version", "reason_code", "severity"],
    "serving.category_summary": ["dataset_version", "category_name", "category_value"],
    "serving.offset_run_config_summary": ["dataset_version", "target_id", "well_id", "bit_type", "bha_type", "mud_type", "mud_weight_ppg", "mud_funnel_viscosity_sec"],
    "serving.offset_run_event_summary": ["dataset_version", "target_id", "well_id", "section_id", "run_id", "event_type", "event_subtype"],
    "serving.planned_target_summary": ["dataset_version", "target_id"],
    "serving.offset_section_summary": ["dataset_version", "target_id", "well_id", "section_id"],
}
RELEASE_TABLE_EXPECTATIONS = RELEASE_PROFILE["expected_release_table_rows"]
PLANNED_TARGET_NON_NULL_MINIMUMS = RELEASE_PROFILE["planned_target_non_null_minimums"]

DATASET_NAME = "drill_run_history"
RELEASE_ID = f"{DATASET_VERSION}:{MANIFEST_SHA256}"
RELEASE_REGISTRY_TABLE = "control.dataset_release_registry"
ACTIVE_RELEASE_POINTER_TABLE = "control.dataset_release_pointer"
PUBLICATION_LOCK_TABLE = "control.dataset_publication_lock"
PUBLICATION_ATTEMPT_TABLE = "control.dataset_publication_attempt"
PUBLICATION_RECOVERY_AUDIT_TABLE = "control.dataset_publication_recovery_audit"
PUBLICATION_EVIDENCE_TABLE = "control.dataset_publication_evidence"
EVIDENCE_SCHEMA_VERSION = "1.0.0"
EVIDENCE_RETENTION_DAYS = 400
MAX_EVIDENCE_JSON_BYTES = 1024 * 1024
ATTEMPT_ID = str(uuid.uuid4())
# Operator-controlled recovery is disabled during normal execution. Recovery is
# explicitly opt-in for one run and requires exact tuple + terminal failed job
# proof before lock ownership can transfer.
DEFAULT_RECOVERY_MODE = False
raw_recovery_mode = globals().get("RECOVERY_MODE", DEFAULT_RECOVERY_MODE)
if isinstance(raw_recovery_mode, bool):
    RECOVERY_MODE = raw_recovery_mode
elif isinstance(raw_recovery_mode, str):
    normalized_recovery_mode = raw_recovery_mode.strip().lower()
    require(
        normalized_recovery_mode in {"true", "false"},
        "INVALID_RECOVERY_MODE_PARAMETER",
        str(raw_recovery_mode),
    )
    RECOVERY_MODE = normalized_recovery_mode == "true"
else:
    raise RuntimeError(
        f"INVALID_RECOVERY_MODE_PARAMETER: {raw_recovery_mode}"
    )


def optional_recovery_parameter(parameter_name: str) -> str:
    raw_value = globals().get(parameter_name, "")
    if raw_value is None:
        return ""
    if isinstance(raw_value, str):
        return raw_value
    raise RuntimeError(
        f"INVALID_{parameter_name}_PARAMETER: {raw_value}"
    )


RECOVERY_OPERATOR = optional_recovery_parameter("RECOVERY_OPERATOR")
RECOVERY_REASON = optional_recovery_parameter("RECOVERY_REASON")
RECOVERY_EXPECTED_DATASET_NAME = optional_recovery_parameter("RECOVERY_EXPECTED_DATASET_NAME")
RECOVERY_EXPECTED_RELEASE_ID = optional_recovery_parameter("RECOVERY_EXPECTED_RELEASE_ID")
RECOVERY_EXPECTED_ATTEMPT_ID = optional_recovery_parameter("RECOVERY_EXPECTED_ATTEMPT_ID")
RECOVERY_EXPECTED_APPLICATION_ID = optional_recovery_parameter("RECOVERY_EXPECTED_APPLICATION_ID")
RECOVERY_EXPECTED_JOB_INSTANCE_ID = optional_recovery_parameter("RECOVERY_EXPECTED_JOB_INSTANCE_ID")
RECOVERY_EXPECTED_JOB_RESPONSE_JSON = optional_recovery_parameter("RECOVERY_EXPECTED_JOB_RESPONSE_JSON")
FABRIC_NOTEBOOK_ITEM_ID = "e7883ed6-638e-4131-9b8e-b6e86aa278b5"
STAGING_STATE = "STAGING"
READY_STATE = "READY"
FAILED_STATE = "FAILED"
FABRIC_FAILED_JOB_STATUS = "Failed"
STAGED_ROW_STATE = "STAGED"
DEFAULT_ACTIVATE_RELEASE_POINTER = True
raw_activate_release_pointer = globals().get("ACTIVATE_RELEASE_POINTER", DEFAULT_ACTIVATE_RELEASE_POINTER)
if isinstance(raw_activate_release_pointer, bool):
    ACTIVATE_RELEASE_POINTER = raw_activate_release_pointer
elif isinstance(raw_activate_release_pointer, str):
    normalized_activate_release_pointer = raw_activate_release_pointer.strip().lower()
    require(
        normalized_activate_release_pointer in {"true", "false"},
        "INVALID_ACTIVATE_RELEASE_POINTER_PARAMETER",
        str(raw_activate_release_pointer),
    )
    ACTIVATE_RELEASE_POINTER = normalized_activate_release_pointer == "true"
else:
    raise RuntimeError(
        f"INVALID_ACTIVATE_RELEASE_POINTER_PARAMETER: {raw_activate_release_pointer}"
    )


DEFAULT_CONTROL_ONLY_PROMOTION = True
raw_control_only_promotion = globals().get("CONTROL_ONLY_PROMOTION", DEFAULT_CONTROL_ONLY_PROMOTION)
if isinstance(raw_control_only_promotion, bool):
    CONTROL_ONLY_PROMOTION = raw_control_only_promotion
elif isinstance(raw_control_only_promotion, str):
    normalized_control_only_promotion = raw_control_only_promotion.strip().lower()
    require(
        normalized_control_only_promotion in {"true", "false"},
        "INVALID_CONTROL_ONLY_PROMOTION_PARAMETER",
        str(raw_control_only_promotion),
    )
    CONTROL_ONLY_PROMOTION = normalized_control_only_promotion == "true"
else:
    raise RuntimeError(
        f"INVALID_CONTROL_ONLY_PROMOTION_PARAMETER: {raw_control_only_promotion}"
    )
require(
    not CONTROL_ONLY_PROMOTION or ACTIVATE_RELEASE_POINTER,
    "CONTROL_ONLY_PROMOTION_REQUIRES_POINTER_ACTIVATION",
    str(ACTIVATE_RELEASE_POINTER),
)
EXECUTION_MODE = (
    "CONTROL_ONLY"
    if CONTROL_ONLY_PROMOTION
    else ("FULL_REBUILD_AND_ACTIVATE" if ACTIVATE_RELEASE_POINTER else "FULL_REBUILD_STAGE_ONLY")
)
pre_mutation_pointer_rows = (
    spark.table(ACTIVE_RELEASE_POINTER_TABLE)
    .filter(F.col("dataset_name") == DATASET_NAME)
    .select("active_release_id")
    .collect()
)
require(
    len(pre_mutation_pointer_rows) <= 1,
    "ACTIVE_RELEASE_POINTER_NOT_UNIQUE",
    DATASET_NAME,
)
pre_mutation_active_release_id = (
    pre_mutation_pointer_rows[0]["active_release_id"] if pre_mutation_pointer_rows else None
)
print(
    json.dumps(
        {
            "stage": "execution_mode",
            "release_version": RELEASE_VERSION,
            "release_id": RELEASE_ID,
            "execution_mode": EXECUTION_MODE,
            "activate_release_pointer": ACTIVATE_RELEASE_POINTER,
            "control_only_promotion": CONTROL_ONLY_PROMOTION,
            "default_control_only_promotion": DEFAULT_CONTROL_ONLY_PROMOTION,
            "pre_mutation_active_release_id": pre_mutation_active_release_id,
            "pre_mutation_release_already_active": pre_mutation_active_release_id == RELEASE_ID,
        },
        sort_keys=True,
    )
)


HASH_EXCLUDED_COLUMNS = {"_processed_at_utc", "_release_id", "_publication_state"}


def canonical_business_columns(columns: list[str]) -> list[str]:
    return sorted(column for column in columns if column not in HASH_EXCLUDED_COLUMNS)


def projection_drift_details(expected_columns: list[str], actual_columns: list[str]) -> dict[str, list[str]]:
    expected_set = set(expected_columns)
    actual_set = set(actual_columns)
    return {
        "expected": list(expected_columns),
        "actual": list(actual_columns),
        "missing": sorted(expected_set - actual_set),
        "extra": sorted(actual_set - expected_set),
    }


def require_expected_business_columns(
    logical_table_name: str,
    frame: DataFrame,
    expected_business_columns: list[str],
    error_code: str,
) -> list[str]:
    actual_business_columns = canonical_business_columns(frame.columns)
    if actual_business_columns != expected_business_columns:
        require(
            False,
            error_code,
            json.dumps(
                {
                    "logical_table": logical_table_name,
                    **projection_drift_details(expected_business_columns, actual_business_columns),
                },
                sort_keys=True,
            ),
        )
    return expected_business_columns


def deterministic_content_hash(
    frame: DataFrame,
    *,
    logical_table_name: str,
    expected_business_columns: list[str] | None = None,
    schema_error_code: str = "CONTENT_HASH_SCHEMA_DRIFT",
) -> str:
    business_columns = canonical_business_columns(frame.columns)
    if expected_business_columns is not None:
        business_columns = require_expected_business_columns(
            logical_table_name,
            frame,
            expected_business_columns,
            schema_error_code,
        )
    result = (
        frame.select(F.to_json(F.struct(*[F.col(column) for column in business_columns])).alias("_row"))
        .agg(F.sha2(F.coalesce(F.concat_ws("||", F.sort_array(F.collect_list("_row"))), F.lit("")), 256).alias("_hash"))
        .collect()[0]["_hash"]
    )
    return result


def physical_table_for(logical_table_name: str) -> str:
    schema_name, table_name = logical_table_name.split(".", 1)
    return f"{schema_name}.__{table_name}_release_data"


def ensure_table_schema_compatible(logical_table_name: str, frame: DataFrame) -> None:
    physical_table_name = physical_table_for(logical_table_name)
    if not spark.catalog.tableExists(physical_table_name):
        return
    existing_columns = {field.name for field in spark.table(physical_table_name).schema.fields}
    missing_fields = [field for field in frame.schema.fields if field.name not in existing_columns]
    if not missing_fields:
        return
    add_columns_sql = ", ".join(
        f"`{field.name}` {field.dataType.simpleString()}" for field in missing_fields
    )
    spark.sql(f"ALTER TABLE {physical_table_name} ADD COLUMNS ({add_columns_sql})")


def sql_literal(value: str) -> str:
    return value.replace("'", "''")


def sanitized_exception_type(error: Exception) -> str:
    return "".join(character for character in type(error).__name__ if character.isalnum() or character == "_")[:128] or "Exception"


def require_recovery_parameter(parameter_name: str, parameter_value: object) -> str:
    value = str(parameter_value).strip() if parameter_value is not None else ""
    require(bool(value), "RECOVERY_PARAMETER_REQUIRED", parameter_name)
    return value


def recovery_job_spark_application_id(job_payload: dict) -> str:
    activity_spark_application_id = (
        job_payload.get("properties", {})
        .get("computeDetails", {})
        .get("monitoringInfo", {})
        .get("activityDetails", {})
        .get("sparkApplicationId")
    )
    require(bool(activity_spark_application_id), "RECOVERY_JOB_PROOF_MISSING_SPARK_APPLICATION", json.dumps(job_payload, sort_keys=True)[:4000])
    return str(activity_spark_application_id)


def recovery_job_summary(job_payload: dict) -> dict[str, str]:
    require(isinstance(job_payload, dict), "RECOVERY_JOB_PROOF_INVALID", str(type(job_payload).__name__))
    status = str(job_payload.get("status") or "")
    require(bool(status), "RECOVERY_JOB_PROOF_MISSING_STATUS", json.dumps(job_payload, sort_keys=True)[:4000])
    activity_details = (
        job_payload.get("properties", {})
        .get("computeDetails", {})
        .get("monitoringInfo", {})
        .get("activityDetails", {})
    )
    creator_item = activity_details.get("creatorItem") or {}
    summary = {
        "id": str(job_payload.get("id") or ""),
        "status": status,
        "workspace_id": str(creator_item.get("workspaceId") or ""),
        "item_id": str(creator_item.get("itemId") or ""),
        "job_item_id": str(job_payload.get("itemId") or ""),
        "root_activity_id": str(job_payload.get("rootActivityId") or ""),
        "end_time_utc": str(job_payload.get("endTimeUtc") or activity_details.get("endDateTime") or ""),
        "failure_error_code": str((job_payload.get("failureReason") or {}).get("errorCode") or ""),
        "spark_application_id": recovery_job_spark_application_id(job_payload),
    }
    require(bool(summary["id"]), "RECOVERY_JOB_PROOF_MISSING_ID", json.dumps(job_payload, sort_keys=True)[:4000])
    require(bool(summary["workspace_id"]), "RECOVERY_JOB_PROOF_MISSING_WORKSPACE", json.dumps(job_payload, sort_keys=True)[:4000])
    require(bool(summary["item_id"]), "RECOVERY_JOB_PROOF_MISSING_ITEM", json.dumps(job_payload, sort_keys=True)[:4000])
    require(bool(summary["job_item_id"]), "RECOVERY_JOB_PROOF_MISSING_JOB_ITEM", json.dumps(job_payload, sort_keys=True)[:4000])
    require(summary["job_item_id"] == summary["item_id"], "RECOVERY_JOB_ITEM_MISMATCH", json.dumps(summary, sort_keys=True))
    require(bool(summary["root_activity_id"]), "RECOVERY_JOB_PROOF_MISSING_ROOT_ACTIVITY", json.dumps(job_payload, sort_keys=True)[:4000])
    require(bool(summary["end_time_utc"]), "RECOVERY_JOB_PROOF_MISSING_END_TIME", json.dumps(job_payload, sort_keys=True)[:4000])
    return summary


def require_terminal_failed_job_status(job_summary: dict[str, str]) -> None:
    require(
        job_summary["status"] == FABRIC_FAILED_JOB_STATUS,
        "RECOVERY_JOB_NOT_TERMINAL_FAILED",
        job_summary["status"],
    )


def fabric_api_access_token() -> str:
    try:
        raw_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    except Exception as token_error:
        raise RuntimeError(f"RECOVERY_TERMINAL_JOB_TOKEN_UNAVAILABLE: {sanitized_exception_type(token_error)}") from token_error

    if isinstance(raw_token, str):
        token = raw_token.strip()
        require(bool(token), "RECOVERY_TERMINAL_JOB_TOKEN_EMPTY", "string")
        return token

    if isinstance(raw_token, dict):
        for token_field in ("accessToken", "token", "access_token"):
            candidate = str(raw_token.get(token_field) or "").strip()
            if candidate:
                return candidate

    raise RuntimeError(f"RECOVERY_TERMINAL_JOB_TOKEN_UNSUPPORTED: {type(raw_token).__name__}")


def fetch_job_instance_record(job_instance_id: str) -> dict:
    import urllib.error
    import urllib.request

    job_url = (
        f"https://api.fabric.microsoft.com/v1/workspaces/{FABRIC_WORKSPACE_ID}"
        f"/notebooks/{FABRIC_NOTEBOOK_ITEM_ID}/jobs/execute/instances/{job_instance_id}"
    )
    request = urllib.request.Request(
        job_url,
        headers={
            "Authorization": f"Bearer {fabric_api_access_token()}",
            "Accept": "application/json",
        },
        method="GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = response.read().decode("utf-8")
    except urllib.error.HTTPError as http_error:
        raise RuntimeError(f"RECOVERY_TERMINAL_JOB_FETCH_HTTP_ERROR: status={http_error.code}") from http_error
    except urllib.error.URLError as url_error:
        raise RuntimeError(f"RECOVERY_TERMINAL_JOB_FETCH_TRANSPORT_ERROR: {type(url_error.reason).__name__}") from url_error

    try:
        parsed_payload = json.loads(payload)
    except json.JSONDecodeError as decode_error:
        raise RuntimeError("RECOVERY_TERMINAL_JOB_FETCH_INVALID_JSON") from decode_error

    return parsed_payload


def assert_publication_lock_owned() -> None:
    owners = (
        spark.table(PUBLICATION_LOCK_TABLE)
        .filter(F.col("dataset_name") == DATASET_NAME)
        .select("attempt_id", "attempt_application_id")
        .collect()
    )
    require(len(owners) == 1, "PUBLICATION_LOCK_SENTINEL_INVALID", DATASET_NAME)
    require(owners[0]["attempt_id"] == ATTEMPT_ID, "PUBLICATION_LOCK_NOT_OWNED", str(owners[0]["attempt_id"]))
    require(owners[0]["attempt_application_id"] == spark.sparkContext.applicationId, "PUBLICATION_LOCK_NOT_OWNED", str(owners[0]["attempt_application_id"]))


def create_active_release_view(logical_table_name: str, frame: DataFrame) -> None:
    physical_table_name = physical_table_for(logical_table_name)
    projected_columns = ", ".join(f"d.`{column}`" for column in frame.columns)
    spark.sql(
        f"""
        CREATE OR REPLACE VIEW {logical_table_name} AS
        SELECT {projected_columns}
        FROM {physical_table_name} d
        INNER JOIN {ACTIVE_RELEASE_POINTER_TABLE} p
          ON p.dataset_name = '{DATASET_NAME}'
         AND p.active_release_id = d._release_id
        INNER JOIN {RELEASE_REGISTRY_TABLE} r
          ON r.dataset_name = p.dataset_name
         AND r.release_id = p.active_release_id
         AND r.release_state = '{READY_STATE}'
        WHERE d._publication_state = '{STAGED_ROW_STATE}'
        """
    )


def require_logical_view_projection(logical_table_name: str, expected_columns: list[str], error_code: str) -> None:
    actual_columns = spark.table(logical_table_name).columns
    if actual_columns != expected_columns:
        require(
            False,
            error_code,
            json.dumps(
                {
                    "logical_table": logical_table_name,
                    **projection_drift_details(expected_columns, actual_columns),
                },
                sort_keys=True,
            ),
        )


EXPECTED_BUSINESS_COLUMNS = {
    table_name: canonical_business_columns(frame.columns)
    for table_name, frame in LOGICAL_TABLES.items()
}

prepublication_hashes = {
    table_name: deterministic_content_hash(
        frame,
        logical_table_name=table_name,
        expected_business_columns=EXPECTED_BUSINESS_COLUMNS[table_name],
        schema_error_code="PREPUBLICATION_CONTENT_HASH_SCHEMA_DRIFT",
    )
    for table_name, frame in LOGICAL_TABLES.items()
}

for schema_name in ("control", "raw", "curated", "serving"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {RELEASE_REGISTRY_TABLE} (
        dataset_name STRING NOT NULL,
        release_id STRING NOT NULL,
        dataset_version STRING NOT NULL,
        manifest_sha256 STRING NOT NULL,
        aggregate_source_sha256 STRING NOT NULL,
        release_state STRING NOT NULL,
        attempt_id STRING NOT NULL,
        attempt_application_id STRING NOT NULL,
        staged_at_utc TIMESTAMP NOT NULL,
        reconciled_at_utc TIMESTAMP,
        failure_message STRING
    ) USING DELTA
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {ACTIVE_RELEASE_POINTER_TABLE} (
        dataset_name STRING NOT NULL,
        active_release_id STRING NOT NULL,
        activated_at_utc TIMESTAMP NOT NULL,
        activated_by STRING NOT NULL
    ) USING DELTA
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {PUBLICATION_LOCK_TABLE}
    USING DELTA AS
    SELECT
        '{DATASET_NAME}' AS dataset_name,
        CAST(NULL AS STRING) AS attempt_id,
        CAST(NULL AS STRING) AS attempt_application_id,
        CAST(NULL AS TIMESTAMP) AS acquired_at_utc
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {PUBLICATION_ATTEMPT_TABLE} (
        dataset_name STRING NOT NULL,
        attempt_id STRING NOT NULL,
        release_id STRING NOT NULL,
        attempt_application_id STRING NOT NULL,
        attempt_state STRING NOT NULL,
        requested_at_utc TIMESTAMP NOT NULL,
        completed_at_utc TIMESTAMP,
        failure_message STRING
    ) USING DELTA
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {PUBLICATION_RECOVERY_AUDIT_TABLE} (
        dataset_name STRING NOT NULL,
        interrupted_attempt_id STRING NOT NULL,
        interrupted_application_id STRING NOT NULL,
        cleared_by STRING NOT NULL,
        recovery_reason STRING NOT NULL,
        terminal_job_confirmed BOOLEAN NOT NULL,
        recorded_at_utc TIMESTAMP NOT NULL
    ) USING DELTA
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {PUBLICATION_EVIDENCE_TABLE} (
        dataset_name STRING NOT NULL,
        attempt_id STRING NOT NULL,
        release_id STRING NOT NULL,
        evidence_schema_version STRING NOT NULL,
        evidence_written_at_utc TIMESTAMP NOT NULL,
        payload_sha256 STRING NOT NULL,
        payload_json STRING NOT NULL
    ) USING DELTA
    TBLPROPERTIES (
        'delta.appendOnly' = 'true',
        'drillrun.evidence.schemaVersion' = '{EVIDENCE_SCHEMA_VERSION}',
        'drillrun.evidence.maxPayloadBytes' = '{MAX_EVIDENCE_JSON_BYTES}',
        'drillrun.evidence.minimumRetentionDays' = '{EVIDENCE_RETENTION_DAYS}'
    )
    """
)
lock_columns = set(spark.table(PUBLICATION_LOCK_TABLE).columns)
require(
    lock_columns == {"dataset_name", "attempt_id", "attempt_application_id", "acquired_at_utc"},
    "NON_EXPIRING_LOCK_SCHEMA_MIGRATION_REQUIRED",
    ",".join(sorted(lock_columns)),
)
evidence_columns = set(spark.table(PUBLICATION_EVIDENCE_TABLE).columns)
require(
    evidence_columns == {
        "dataset_name", "attempt_id", "release_id", "evidence_schema_version",
        "evidence_written_at_utc", "payload_sha256", "payload_json",
    },
    "EVIDENCE_SCHEMA_MIGRATION_REQUIRED",
    ",".join(sorted(evidence_columns)),
)

legacy_direct_objects = []
for logical_table_name in LOGICAL_TABLES:
    if spark.catalog.tableExists(logical_table_name):
        object_type = spark.catalog.getTable(logical_table_name).tableType.upper()
        if object_type != "VIEW":
            legacy_direct_objects.append(f"{logical_table_name}:{object_type}")
require(not legacy_direct_objects, "LEGACY_DIRECT_TABLE_REQUIRES_ONE_TIME_MIGRATION", ",".join(legacy_direct_objects))

application_literal = sql_literal(spark.sparkContext.applicationId)
attempt_literal = sql_literal(ATTEMPT_ID)
release_literal = sql_literal(RELEASE_ID)
execution_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]

lock_sentinel_rows = (
    spark.table(PUBLICATION_LOCK_TABLE)
    .filter(F.col("dataset_name") == DATASET_NAME)
    .select("attempt_id", "attempt_application_id")
    .collect()
)
require(len(lock_sentinel_rows) == 1, "PUBLICATION_LOCK_SENTINEL_INVALID", DATASET_NAME)
existing_locks = (
    spark.table(PUBLICATION_LOCK_TABLE)
    .filter((F.col("dataset_name") == DATASET_NAME) & F.col("attempt_id").isNotNull())
    .select("attempt_id", "attempt_application_id")
    .collect()
)
require(len(existing_locks) <= 1, "PUBLICATION_LOCK_SENTINEL_INVALID", DATASET_NAME)
lock_claimed_by_recovery_transfer = False
if RECOVERY_MODE:
    require(len(existing_locks) == 1, "RECOVERY_LOCK_REQUIRED", DATASET_NAME)
else:
    recovery_override_values = [
        RECOVERY_OPERATOR,
        RECOVERY_REASON,
        RECOVERY_EXPECTED_DATASET_NAME,
        RECOVERY_EXPECTED_RELEASE_ID,
        RECOVERY_EXPECTED_ATTEMPT_ID,
        RECOVERY_EXPECTED_APPLICATION_ID,
        RECOVERY_EXPECTED_JOB_INSTANCE_ID,
        RECOVERY_EXPECTED_JOB_RESPONSE_JSON,
    ]
    require(
        not any(str(value).strip() for value in recovery_override_values if value is not None),
        "RECOVERY_PARAMETERS_REQUIRE_RECOVERY_MODE",
        DATASET_NAME,
    )
spark.sql(
    f"""
    INSERT INTO {PUBLICATION_ATTEMPT_TABLE}
    VALUES (
        '{DATASET_NAME}', '{attempt_literal}', '{release_literal}',
        '{application_literal}', 'REQUESTED', current_timestamp(), NULL, NULL
    )
    """
)
if existing_locks:
    existing_lock = existing_locks[0]
    active_before_recovery = (
        spark.table(ACTIVE_RELEASE_POINTER_TABLE)
        .filter(F.col("dataset_name") == DATASET_NAME)
        .select("active_release_id")
        .collect()
    )
    if not RECOVERY_MODE:
        error_code = "ACTIVE_RELEASE_LOCK_CLEANUP_REQUIRED" if active_before_recovery and active_before_recovery[0]["active_release_id"] == RELEASE_ID else "PUBLICATION_LOCK_HELD_NO_AUTOMATIC_TAKEOVER"
        raise RuntimeError(f"{error_code}: attempt_id={existing_lock['attempt_id']} application_id={existing_lock['attempt_application_id']}")

    require(bool(RECOVERY_OPERATOR.strip()), "RECOVERY_OPERATOR_REQUIRED", DATASET_NAME)
    require(bool(RECOVERY_REASON.strip()), "RECOVERY_REASON_REQUIRED", DATASET_NAME)

    expected_dataset_name = require_recovery_parameter("RECOVERY_EXPECTED_DATASET_NAME", RECOVERY_EXPECTED_DATASET_NAME)
    expected_release_id = require_recovery_parameter("RECOVERY_EXPECTED_RELEASE_ID", RECOVERY_EXPECTED_RELEASE_ID)
    expected_attempt_id = require_recovery_parameter("RECOVERY_EXPECTED_ATTEMPT_ID", RECOVERY_EXPECTED_ATTEMPT_ID)
    expected_application_id = require_recovery_parameter("RECOVERY_EXPECTED_APPLICATION_ID", RECOVERY_EXPECTED_APPLICATION_ID)
    expected_job_instance_id = require_recovery_parameter("RECOVERY_EXPECTED_JOB_INSTANCE_ID", RECOVERY_EXPECTED_JOB_INSTANCE_ID)
    expected_job_response_json = require_recovery_parameter("RECOVERY_EXPECTED_JOB_RESPONSE_JSON", RECOVERY_EXPECTED_JOB_RESPONSE_JSON)

    require(expected_dataset_name == DATASET_NAME, "RECOVERY_DATASET_NAME_MISMATCH", expected_dataset_name)
    require(existing_lock["attempt_id"] == expected_attempt_id, "RECOVERY_LOCK_ATTEMPT_MISMATCH", str(existing_lock["attempt_id"]))
    require(existing_lock["attempt_application_id"] == expected_application_id, "RECOVERY_LOCK_APPLICATION_MISMATCH", str(existing_lock["attempt_application_id"]))

    recovered_attempt = (
        spark.table(PUBLICATION_ATTEMPT_TABLE)
        .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("attempt_id") == expected_attempt_id))
        .select("release_id", "attempt_state", "attempt_application_id")
        .collect()
    )
    require(len(recovered_attempt) == 1, "RECOVERY_ATTEMPT_ROW_MISSING", expected_attempt_id)
    require(recovered_attempt[0]["release_id"] == expected_release_id, "RECOVERY_ATTEMPT_RELEASE_MISMATCH", str(recovered_attempt[0]["release_id"]))
    require(recovered_attempt[0]["attempt_application_id"] == expected_application_id, "RECOVERY_ATTEMPT_APPLICATION_MISMATCH", str(recovered_attempt[0]["attempt_application_id"]))
    require(recovered_attempt[0]["attempt_state"] == FAILED_STATE, "RECOVERY_ATTEMPT_NOT_FAILED", str(recovered_attempt[0]["attempt_state"]))

    try:
        operator_job_payload = json.loads(expected_job_response_json)
    except json.JSONDecodeError as decode_error:
        raise RuntimeError("RECOVERY_JOB_PROOF_INVALID_JSON") from decode_error

    operator_job_summary = recovery_job_summary(operator_job_payload)
    require(operator_job_summary["id"] == expected_job_instance_id, "RECOVERY_JOB_PROOF_ID_MISMATCH", operator_job_summary["id"])
    require_terminal_failed_job_status(operator_job_summary)
    require(operator_job_summary["workspace_id"] == FABRIC_WORKSPACE_ID, "RECOVERY_JOB_WORKSPACE_MISMATCH", operator_job_summary["workspace_id"])
    require(operator_job_summary["item_id"] == FABRIC_NOTEBOOK_ITEM_ID, "RECOVERY_JOB_ITEM_MISMATCH", operator_job_summary["item_id"])
    require(operator_job_summary["spark_application_id"] == expected_application_id, "RECOVERY_TERMINAL_JOB_CONFIRMATION_MISMATCH", operator_job_summary["spark_application_id"])

    trusted_job_payload = fetch_job_instance_record(expected_job_instance_id)
    trusted_job_summary = recovery_job_summary(trusted_job_payload)
    require_terminal_failed_job_status(trusted_job_summary)
    require(trusted_job_summary["spark_application_id"] == expected_application_id, "RECOVERY_TERMINAL_JOB_CONFIRMATION_MISMATCH", trusted_job_summary["spark_application_id"])
    require(
        trusted_job_summary == operator_job_summary,
        "RECOVERY_JOB_PROOF_MISMATCH",
        json.dumps({"trusted": trusted_job_summary, "operator": operator_job_summary}, sort_keys=True),
    )

    interrupted_attempt_literal = sql_literal(expected_attempt_id)
    interrupted_application_literal = sql_literal(expected_application_id)
    spark.sql(
        f"""
        INSERT INTO {PUBLICATION_RECOVERY_AUDIT_TABLE}
        VALUES (
            '{DATASET_NAME}', '{interrupted_attempt_literal}',
            '{interrupted_application_literal}', '{sql_literal(RECOVERY_OPERATOR)}',
            '{sql_literal(RECOVERY_REASON)}', true, current_timestamp()
        )
        """
    )
    spark.sql(
        f"""
        UPDATE {PUBLICATION_LOCK_TABLE}
        SET attempt_id = '{attempt_literal}',
            attempt_application_id = '{application_literal}',
            acquired_at_utc = current_timestamp()
        WHERE dataset_name = '{DATASET_NAME}'
          AND attempt_id = '{interrupted_attempt_literal}'
          AND attempt_application_id = '{interrupted_application_literal}'
        """
    )
    assert_publication_lock_owned()
    lock_claimed_by_recovery_transfer = True
# This conditional update of the one pre-created dataset sentinel is the only
# lock acquisition transaction in normal execution. Recovery mode uses a bounded
# compare-and-set transfer from an exact failed attempt/application tuple.
if not lock_claimed_by_recovery_transfer:
    spark.sql(
        f"""
        UPDATE {PUBLICATION_LOCK_TABLE}
        SET attempt_id = '{attempt_literal}',
            attempt_application_id = '{application_literal}',
            acquired_at_utc = current_timestamp()
        WHERE dataset_name = '{DATASET_NAME}'
          AND attempt_id IS NULL
        """
    )
    assert_publication_lock_owned()
spark.sql(
    f"""
    UPDATE {PUBLICATION_ATTEMPT_TABLE}
    SET attempt_state = 'LOCKED'
    WHERE dataset_name = '{DATASET_NAME}'
      AND attempt_id = '{attempt_literal}'
    """
)

active_pointer_rows = (
    spark.table(ACTIVE_RELEASE_POINTER_TABLE)
    .filter(F.col("dataset_name") == DATASET_NAME)
    .select("active_release_id")
    .collect()
)
require(len(active_pointer_rows) <= 1, "ACTIVE_RELEASE_POINTER_NOT_UNIQUE", DATASET_NAME)
active_release_id = active_pointer_rows[0]["active_release_id"] if active_pointer_rows else None
release_already_active = active_release_id == RELEASE_ID
staged_tables = {}
release_gate_tables = {}
activation_performed = False
ready_transition_performed = False

def capture_verified_staged_table(logical_table_name: str, row_count_error_code: str, content_hash_error_code: str, key_violation_prefix: str) -> dict:
    physical_table_name = physical_table_for(logical_table_name)
    require(spark.catalog.tableExists(physical_table_name), "ACTIVE_RELEASE_TABLE_MISSING", physical_table_name)
    staged = (
        spark.table(physical_table_name)
        .filter(
            (F.col("_release_id") == RELEASE_ID)
            & (F.col("_publication_state") == STAGED_ROW_STATE)
        )
    )
    actual_rows = staged.count()
    require(
        actual_rows == EXPECTED_TABLE_ROWS[logical_table_name],
        row_count_error_code,
        f"{logical_table_name} expected={EXPECTED_TABLE_ROWS[logical_table_name]} actual={actual_rows}",
    )
    require_unique(staged, TABLE_KEYS[logical_table_name], f"{key_violation_prefix}_{logical_table_name.upper().replace('.', '_')}_KEY_VIOLATION")
    post_hash = deterministic_content_hash(
        staged,
        logical_table_name=logical_table_name,
        expected_business_columns=EXPECTED_BUSINESS_COLUMNS[logical_table_name],
        schema_error_code=f"{content_hash_error_code}_SCHEMA_DRIFT",
    )
    require(
        post_hash == prepublication_hashes[logical_table_name],
        content_hash_error_code,
        json.dumps(
            {
                "logical_table": logical_table_name,
                "expected_sha256": prepublication_hashes[logical_table_name],
                "actual_sha256": post_hash,
                "business_columns": EXPECTED_BUSINESS_COLUMNS[logical_table_name],
            },
            sort_keys=True,
        ),
    )
    history = spark.sql(f"DESCRIBE HISTORY {physical_table_name} LIMIT 1").collect()[0]
    return {
        "physical_table": physical_table_name,
        "row_count": actual_rows,
        "delta_version": int(history["version"]),
        "business_content_sha256": post_hash,
    }

if not release_already_active:
    assert_publication_lock_owned()
    if CONTROL_ONLY_PROMOTION:
        registry_row = (
            spark.table(RELEASE_REGISTRY_TABLE)
            .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("release_id") == RELEASE_ID))
            .select("release_state", "manifest_sha256", "aggregate_source_sha256")
            .collect()
        )
        require(len(registry_row) == 1, "CONTROL_ONLY_REGISTRY_ROW_MISSING", RELEASE_ID)
        require(registry_row[0]["release_state"] == STAGING_STATE, "CONTROL_ONLY_RELEASE_NOT_STAGING", str(registry_row[0]["release_state"]))
        require(registry_row[0]["manifest_sha256"] == MANIFEST_SHA256, "CONTROL_ONLY_MANIFEST_MISMATCH", RELEASE_ID)
        require(registry_row[0]["aggregate_source_sha256"] == AGGREGATE_SOURCE_SHA256, "CONTROL_ONLY_SOURCE_HASH_MISMATCH", RELEASE_ID)
        spark.sql(
            f"""
            UPDATE {RELEASE_REGISTRY_TABLE}
            SET attempt_id = '{attempt_literal}',
                attempt_application_id = '{application_literal}',
                failure_message = NULL
            WHERE dataset_name = '{DATASET_NAME}'
              AND release_id = '{release_literal}'
              AND release_state = '{STAGING_STATE}'
              AND manifest_sha256 = '{MANIFEST_SHA256}'
              AND aggregate_source_sha256 = '{AGGREGATE_SOURCE_SHA256}'
            """
        )
        registry_claimed = (
            spark.table(RELEASE_REGISTRY_TABLE)
            .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("release_id") == RELEASE_ID))
            .select("release_state", "attempt_id", "attempt_application_id")
            .collect()
        )
        require(len(registry_claimed) == 1, "CONTROL_ONLY_REGISTRY_ROW_MISSING", RELEASE_ID)
        require(registry_claimed[0]["release_state"] == STAGING_STATE, "CONTROL_ONLY_RELEASE_NOT_STAGING", str(registry_claimed[0]["release_state"]))
        require(registry_claimed[0]["attempt_id"] == ATTEMPT_ID, "CONTROL_ONLY_REGISTRY_ATTEMPT_MISMATCH", str(registry_claimed[0]["attempt_id"]))
        require(registry_claimed[0]["attempt_application_id"] == spark.sparkContext.applicationId, "CONTROL_ONLY_REGISTRY_APPLICATION_MISMATCH", str(registry_claimed[0]["attempt_application_id"]))
    else:
        spark.sql(
            f"""
            MERGE INTO {RELEASE_REGISTRY_TABLE} target
            USING (
                SELECT
                    '{DATASET_NAME}' AS dataset_name,
                    '{release_literal}' AS release_id,
                    '{DATASET_VERSION}' AS dataset_version,
                    '{MANIFEST_SHA256}' AS manifest_sha256,
                    '{AGGREGATE_SOURCE_SHA256}' AS aggregate_source_sha256,
                    '{STAGING_STATE}' AS release_state,
                    '{attempt_literal}' AS attempt_id,
                    '{application_literal}' AS attempt_application_id,
                    current_timestamp() AS staged_at_utc
            ) source
            ON target.dataset_name = source.dataset_name
           AND target.release_id = source.release_id
            WHEN MATCHED AND target.release_state <> '{READY_STATE}' THEN UPDATE SET
                dataset_version = source.dataset_version,
                manifest_sha256 = source.manifest_sha256,
                aggregate_source_sha256 = source.aggregate_source_sha256,
                release_state = source.release_state,
                attempt_id = source.attempt_id,
                attempt_application_id = source.attempt_application_id,
                staged_at_utc = source.staged_at_utc,
                reconciled_at_utc = NULL,
                failure_message = NULL
            WHEN NOT MATCHED THEN INSERT (
                dataset_name, release_id, dataset_version, manifest_sha256,
                aggregate_source_sha256, release_state, attempt_id, attempt_application_id,
                staged_at_utc, reconciled_at_utc, failure_message
            ) VALUES (
                source.dataset_name, source.release_id, source.dataset_version,
                source.manifest_sha256, source.aggregate_source_sha256,
                source.release_state, source.attempt_id, source.attempt_application_id,
                source.staged_at_utc, NULL, NULL
            )
            """
        )
        registry_row = (
            spark.table(RELEASE_REGISTRY_TABLE)
            .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("release_id") == RELEASE_ID))
            .select("release_state", "manifest_sha256", "aggregate_source_sha256")
            .collect()
        )
        require(len(registry_row) == 1, "STAGING_REGISTRY_ROW_MISSING", RELEASE_ID)
        require(registry_row[0]["release_state"] == STAGING_STATE, "RELEASE_NOT_INACTIVE_BEFORE_WRITES", str(registry_row[0]["release_state"]))
        require(registry_row[0]["manifest_sha256"] == MANIFEST_SHA256, "STAGING_MANIFEST_MISMATCH", RELEASE_ID)
        require(registry_row[0]["aggregate_source_sha256"] == AGGREGATE_SOURCE_SHA256, "STAGING_SOURCE_HASH_MISMATCH", RELEASE_ID)

    try:
        if not CONTROL_ONLY_PROMOTION:
            for logical_table_name, frame in LOGICAL_TABLES.items():
                assert_publication_lock_owned()
                ensure_table_schema_compatible(logical_table_name, frame)
                physical_table_name = physical_table_for(logical_table_name)
                if spark.catalog.tableExists(physical_table_name):
                    spark.sql(f"DELETE FROM {physical_table_name} WHERE _release_id = '{release_literal}'")

            for logical_table_name, frame in LOGICAL_TABLES.items():
                assert_publication_lock_owned()
                physical_table_name = physical_table_for(logical_table_name)
                staged_frame = (
                    frame.withColumn("_release_id", F.lit(RELEASE_ID))
                    .withColumn("_publication_state", F.lit(STAGED_ROW_STATE))
                )
                writer = staged_frame.write.format("delta").mode("append")
                if spark.catalog.tableExists(physical_table_name):
                    writer.saveAsTable(physical_table_name)
                else:
                    writer.partitionBy("_release_id").saveAsTable(physical_table_name)

            for logical_table_name, expected_rows in EXPECTED_TABLE_ROWS.items():
                physical_table_name = physical_table_for(logical_table_name)
                staged = (
                    spark.table(physical_table_name)
                    .filter(
                        (F.col("_release_id") == RELEASE_ID)
                        & (F.col("_publication_state") == STAGED_ROW_STATE)
                    )
                )
                actual_rows = staged.count()
                require(actual_rows == expected_rows, "STAGED_ROW_COUNT_MISMATCH", f"{logical_table_name} expected={expected_rows} actual={actual_rows}")
                require_unique(staged, TABLE_KEYS[logical_table_name], f"STAGED_{logical_table_name.upper().replace('.', '_')}_KEY_VIOLATION")
                post_hash = deterministic_content_hash(
                    staged,
                    logical_table_name=logical_table_name,
                    expected_business_columns=EXPECTED_BUSINESS_COLUMNS[logical_table_name],
                    schema_error_code="STAGED_CONTENT_HASH_SCHEMA_DRIFT",
                )
                require(
                    post_hash == prepublication_hashes[logical_table_name],
                    "STAGED_CONTENT_HASH_MISMATCH",
                    json.dumps(
                        {
                            "logical_table": logical_table_name,
                            "expected_sha256": prepublication_hashes[logical_table_name],
                            "actual_sha256": post_hash,
                            "business_columns": EXPECTED_BUSINESS_COLUMNS[logical_table_name],
                        },
                        sort_keys=True,
                    ),
                )
                history = spark.sql(f"DESCRIBE HISTORY {physical_table_name} LIMIT 1").collect()[0]
                staged_tables[logical_table_name] = {
                    "physical_table": physical_table_name,
                    "row_count": actual_rows,
                    "delta_version": int(history["version"]),
                    "business_content_sha256": post_hash,
                }

        for physical_table_name, expected_rows in RELEASE_TABLE_EXPECTATIONS.items():
            assert_publication_lock_owned()
            require(spark.catalog.tableExists(physical_table_name), "RELEASE_GATE_TABLE_MISSING", physical_table_name)
            staged_release_data = (
                spark.table(physical_table_name)
                .filter(
                    (F.col("_release_id") == RELEASE_ID)
                    & (F.col("_publication_state") == STAGED_ROW_STATE)
                )
            )
            actual_rows = staged_release_data.count()
            require(
                actual_rows == expected_rows,
                "RELEASE_GATE_ROW_COUNT_MISMATCH",
                f"{physical_table_name} expected={expected_rows} actual={actual_rows}",
            )
            history = spark.sql(f"DESCRIBE HISTORY {physical_table_name} LIMIT 1").collect()[0]
            release_gate_tables[physical_table_name] = {
                "physical_table": physical_table_name,
                "row_count": actual_rows,
                "delta_version": int(history["version"]),
                "business_content_sha256": deterministic_content_hash(
                    staged_release_data,
                    logical_table_name=physical_table_name,
                ),
            }

        planned_staged = (
            spark.table("serving.__planned_target_summary_release_data")
            .filter(
                (F.col("_release_id") == RELEASE_ID)
                & (F.col("_publication_state") == STAGED_ROW_STATE)
            )
        )
        for field_name, minimum_required in PLANNED_TARGET_NON_NULL_MINIMUMS.items():
            populated_rows = planned_staged.filter(
                F.col(field_name).isNotNull() & (F.length(F.trim(F.col(field_name))) > 0)
            ).count()
            require(
                populated_rows >= minimum_required,
                "PLANNED_TARGET_FIELD_MINIMUM_NOT_MET",
                f"{field_name} minimum={minimum_required} actual={populated_rows}",
            )

        for logical_table_name, frame in LOGICAL_TABLES.items():
            create_active_release_view(logical_table_name, frame)
            require_logical_view_projection(
                logical_table_name,
                frame.columns,
                "ACTIVE_VIEW_PROJECTION_DRIFT",
            )

        if CONTROL_ONLY_PROMOTION:
            for logical_table_name in EXPECTED_TABLE_ROWS:
                assert_publication_lock_owned()
                staged_tables[logical_table_name] = capture_verified_staged_table(
                    logical_table_name,
                    "CONTROL_ONLY_ACTIVE_RELEASE_ROW_COUNT_MISMATCH",
                    "CONTROL_ONLY_ACTIVE_RELEASE_CONTENT_HASH_MISMATCH",
                    "CONTROL_ONLY_ACTIVE",
                )

        RECONCILIATION_COMPLETE_BEFORE_ACTIVATION = True
        require(RECONCILIATION_COMPLETE_BEFORE_ACTIVATION, "RECONCILIATION_FAILED", RELEASE_ID)
    except Exception as staging_error:
        assert_publication_lock_owned()
        failure_literal = sql_literal(str(staging_error)[:4000])
        spark.sql(
            f"""
            UPDATE {RELEASE_REGISTRY_TABLE}
            SET release_state = '{FAILED_STATE}',
                failure_message = '{failure_literal}'
            WHERE dataset_name = '{DATASET_NAME}'
              AND release_id = '{release_literal}'
              AND attempt_id = '{attempt_literal}'
              AND release_state <> '{READY_STATE}'
            """
        )
        spark.sql(
            f"""
            UPDATE {PUBLICATION_ATTEMPT_TABLE}
            SET attempt_state = '{FAILED_STATE}',
                completed_at_utc = current_timestamp(),
                failure_message = '{failure_literal}'
            WHERE dataset_name = '{DATASET_NAME}'
              AND attempt_id = '{attempt_literal}'
            """
        )
        raise RuntimeError(f"PUBLICATION_FAILED_LOCK_RETAINED: {staging_error}") from staging_error
else:
    assert_publication_lock_owned()
    for logical_table_name, frame in LOGICAL_TABLES.items():
        assert_publication_lock_owned()
        physical_table_name = physical_table_for(logical_table_name)
        require(spark.catalog.tableExists(physical_table_name), "ACTIVE_RELEASE_TABLE_MISSING", physical_table_name)
        staged = (
            spark.table(physical_table_name)
            .filter(
                (F.col("_release_id") == RELEASE_ID)
                & (F.col("_publication_state") == STAGED_ROW_STATE)
            )
        )
        actual_rows = staged.count()
        require(actual_rows == EXPECTED_TABLE_ROWS[logical_table_name], "ACTIVE_RELEASE_ROW_COUNT_MISMATCH", logical_table_name)
        require_unique(staged, TABLE_KEYS[logical_table_name], f"ACTIVE_{logical_table_name.upper().replace('.', '_')}_KEY_VIOLATION")
        post_hash = deterministic_content_hash(
            staged,
            logical_table_name=logical_table_name,
            expected_business_columns=EXPECTED_BUSINESS_COLUMNS[logical_table_name],
            schema_error_code="ACTIVE_RELEASE_CONTENT_HASH_SCHEMA_DRIFT",
        )
        require(
            post_hash == prepublication_hashes[logical_table_name],
            "ACTIVE_RELEASE_CONTENT_HASH_MISMATCH",
            json.dumps(
                {
                    "logical_table": logical_table_name,
                    "expected_sha256": prepublication_hashes[logical_table_name],
                    "actual_sha256": post_hash,
                    "business_columns": EXPECTED_BUSINESS_COLUMNS[logical_table_name],
                },
                sort_keys=True,
            ),
        )
        history = spark.sql(f"DESCRIBE HISTORY {physical_table_name} LIMIT 1").collect()[0]
        staged_tables[logical_table_name] = {
            "physical_table": physical_table_name,
            "row_count": actual_rows,
            "delta_version": int(history["version"]),
            "business_content_sha256": post_hash,
        }
        create_active_release_view(logical_table_name, frame)

active_pointer = (
    spark.table(ACTIVE_RELEASE_POINTER_TABLE)
    .filter(F.col("dataset_name") == DATASET_NAME)
    .select("active_release_id")
    .collect()
)
if ACTIVATE_RELEASE_POINTER and release_already_active:
    require(len(active_pointer) == 1, "ACTIVATION_POINTER_MISSING", DATASET_NAME)
    require(active_pointer[0]["active_release_id"] == RELEASE_ID, "ACTIVATION_POINTER_MISMATCH", str(active_pointer[0]["active_release_id"]))

    for logical_table_name, expected_rows in EXPECTED_TABLE_ROWS.items():
        assert_publication_lock_owned()
        active_view = spark.table(logical_table_name)
        actual_rows = active_view.count()
        require(actual_rows == expected_rows, "ACTIVE_VIEW_ROW_COUNT_MISMATCH", f"{logical_table_name} expected={expected_rows} actual={actual_rows}")
        require_unique(active_view, TABLE_KEYS[logical_table_name], f"ACTIVE_VIEW_{logical_table_name.upper().replace('.', '_')}_KEY_VIOLATION")
        active_view_hash = deterministic_content_hash(
            active_view,
            logical_table_name=logical_table_name,
            expected_business_columns=EXPECTED_BUSINESS_COLUMNS[logical_table_name],
            schema_error_code="ACTIVE_VIEW_CONTENT_HASH_SCHEMA_DRIFT",
        )
        require(
            active_view_hash == prepublication_hashes[logical_table_name],
            "ACTIVE_VIEW_CONTENT_HASH_MISMATCH",
            json.dumps(
                {
                    "logical_table": logical_table_name,
                    "expected_sha256": prepublication_hashes[logical_table_name],
                    "actual_sha256": active_view_hash,
                    "business_columns": EXPECTED_BUSINESS_COLUMNS[logical_table_name],
                },
                sort_keys=True,
            ),
        )
elif not ACTIVATE_RELEASE_POINTER:
    require(
        len(active_pointer) <= 1 and (not active_pointer or active_pointer[0]["active_release_id"] == active_release_id),
        "STAGE_ONLY_POINTER_UNEXPECTED",
        str(active_pointer[0]["active_release_id"] if active_pointer else None),
    )

# Every operation needed to create, persist, and verify evidence is fail-closed.
# Any exception in this boundary leaves the non-expiring lock owned by this attempt.
try:
    assert_publication_lock_owned()
    pointer_history = spark.sql(f"DESCRIBE HISTORY {ACTIVE_RELEASE_POINTER_TABLE} LIMIT 1").collect()[0]
    quarantine_evidence = [
        {
            "run_id": row["run_id"],
            "quality_issue_id": row["quality_issue_id"],
            "reason_code": row["quarantine_reason_code"],
        }
        for row in curated_run_quarantine.select(
            "run_id", "quality_issue_id", "quarantine_reason_code"
        ).orderBy("run_id").collect()
    ]
    require(len(quarantine_evidence) == 5, "QUARANTINE_EVIDENCE_COUNT_MISMATCH", str(len(quarantine_evidence)))
    evidence_written_at = datetime.now(timezone.utc)
    reconciliation_result = {
        "evidence_contract": {
            "schema_version": EVIDENCE_SCHEMA_VERSION,
            "retention_days": EVIDENCE_RETENTION_DAYS,
            "max_payload_bytes": MAX_EVIDENCE_JSON_BYTES,
            "canonicalization": "UTF-8 JSON; keys sorted; separators comma/colon; ensure_ascii=false",
            "integrity": {
                "algorithm": "sha256",
                "field": "evidence_contract.integrity.content_sha256",
                "verification": "Set content_sha256 to null, canonicalize the complete payload, then hash the UTF-8 bytes",
                "content_sha256": None,
            },
        },
        "evidence_written_at_utc": evidence_written_at.isoformat(),
        "result": "PASS",
        "contract_version": CONTRACT_VERSION,
        "dataset": {
            "name": DATASET_NAME,
            "version": DATASET_VERSION,
            "release_id": RELEASE_ID,
            "is_synthetic": True,
            "dataset_fingerprint": {"algorithm": "sha256", "value": MANIFEST_SHA256},
        },
        "authorized_inputs": {
            "source_path": SOURCE_ROOT,
            "manifest_sha256": MANIFEST_SHA256,
            "source_inventory_sha256": SOURCE_INVENTORY_SHA256,
            "aggregate_source_fingerprint": {"algorithm": "sha256", "value": AGGREGATE_SOURCE_SHA256},
            "authorized_file_sha256": dict(sorted(EXPECTED_FILE_HASHES.items())),
            "observed_file_sha256": dict(sorted(actual_file_hashes.items())),
            "prior_load_evidence_contract": PRIOR_LOAD_EVIDENCE_CONTRACT,
        },
        "row_counts": {
            "source_files": dict(sorted(actual_source_rows.items())),
            "raw": {name: details["row_count"] for name, details in sorted(staged_tables.items()) if name.startswith("raw.")},
            "curated": {name: details["row_count"] for name, details in sorted(staged_tables.items()) if name.startswith("curated.")},
            "serving": {name: details["row_count"] for name, details in sorted(staged_tables.items()) if name.startswith("serving.")},
            "release_data_gates": {name: details["row_count"] for name, details in sorted(release_gate_tables.items())},
        },
        "routing": {
            "source_run_count": staged_tables["raw.drilling_run"]["row_count"],
            "accepted_run_count": staged_tables["curated.drilling_run"]["row_count"],
            "quarantined_run_count": staged_tables["curated.drilling_run_quarantine"]["row_count"],
            "rejected_record_count": staged_tables["curated.rejected_record"]["row_count"],
            "accepted_evidence_table": "curated.drilling_run",
            "quarantine_evidence_table": "curated.drilling_run_quarantine",
            "rejected_evidence_table": "curated.rejected_record",
            "quarantine_issues": quarantine_evidence,
        },
        "summary_reconciliation_measures": {
            key: str(actual_dataset_summary[key]) if isinstance(actual_dataset_summary[key], Decimal) else actual_dataset_summary[key]
            for key in sorted(expected_dataset_summary)
        },
        "physical_tables": dict(sorted(staged_tables.items())),
        "publication": {
            "release_already_active": release_already_active,
            "activation_enabled": ACTIVATE_RELEASE_POINTER,
            "control_only_promotion": CONTROL_ONLY_PROMOTION,
            "activation_performed": activation_performed,
            "activation_result": "CONTROL_ONLY_ACTIVATED" if (activation_performed and CONTROL_ONLY_PROMOTION) else ("ACTIVATED" if activation_performed else ("STAGED_ONLY_NO_POINTER_UPDATE" if not ACTIVATE_RELEASE_POINTER else "ALREADY_ACTIVE_VERIFIED")),
            "active_release_id": active_release_id if not ACTIVATE_RELEASE_POINTER else RELEASE_ID,
            "pointer_table": ACTIVE_RELEASE_POINTER_TABLE,
            "pointer_delta_version": int(pointer_history["version"]),
        },
        "execution": {
            "attempt_id": ATTEMPT_ID,
            "generation_run_id": GENERATION_RUN_ID,
            "spark_application_id": spark.sparkContext.applicationId,
            "user": execution_user,
            "workspace_id": FABRIC_WORKSPACE_ID,
            "lakehouse_id": FABRIC_LAKEHOUSE_ID,
            "lakehouse_name": FABRIC_LAKEHOUSE_NAME,
            "notebook_name": FABRIC_NOTEBOOK_NAME,
            "processed_at_utc": PROCESSED_AT_UTC.isoformat(),
            "completed_at_utc": evidence_written_at.isoformat(),
        },
    }
    integrity_content_json = json.dumps(reconciliation_result, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    integrity_content_sha256 = hashlib.sha256(integrity_content_json.encode("utf-8")).hexdigest()
    reconciliation_result["evidence_contract"]["integrity"]["content_sha256"] = integrity_content_sha256
    reconciliation_json = json.dumps(reconciliation_result, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    reconciliation_json_bytes = reconciliation_json.encode("utf-8")
    require(
        len(reconciliation_json_bytes) <= MAX_EVIDENCE_JSON_BYTES,
        "EVIDENCE_PAYLOAD_TOO_LARGE",
        f"bytes={len(reconciliation_json_bytes)} max={MAX_EVIDENCE_JSON_BYTES}",
    )
    evidence_payload_sha256 = hashlib.sha256(reconciliation_json_bytes).hexdigest()
    evidence_timestamp_literal = evidence_written_at.strftime("%Y-%m-%d %H:%M:%S.%f")
    assert_publication_lock_owned()
    spark.sql(
        f"""
        INSERT INTO {PUBLICATION_EVIDENCE_TABLE} (
            dataset_name, attempt_id, release_id, evidence_schema_version,
            evidence_written_at_utc, payload_sha256, payload_json
        ) VALUES (
            '{DATASET_NAME}', '{attempt_literal}', '{release_literal}',
            '{EVIDENCE_SCHEMA_VERSION}', CAST('{evidence_timestamp_literal}' AS TIMESTAMP),
            '{evidence_payload_sha256}', '{sql_literal(reconciliation_json)}'
        )
        """
    )
    assert_publication_lock_owned()
    persisted_evidence = (
        spark.table(PUBLICATION_EVIDENCE_TABLE)
        .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("attempt_id") == ATTEMPT_ID))
        .select("release_id", "evidence_schema_version", "payload_sha256", "payload_json")
        .collect()
    )
    require(len(persisted_evidence) == 1, "PUBLICATION_EVIDENCE_NOT_UNIQUE", ATTEMPT_ID)
    require(persisted_evidence[0]["release_id"] == RELEASE_ID, "PUBLICATION_EVIDENCE_RELEASE_MISMATCH", ATTEMPT_ID)
    require(persisted_evidence[0]["evidence_schema_version"] == EVIDENCE_SCHEMA_VERSION, "PUBLICATION_EVIDENCE_SCHEMA_MISMATCH", ATTEMPT_ID)
    require(persisted_evidence[0]["payload_sha256"] == evidence_payload_sha256, "PUBLICATION_EVIDENCE_HASH_MISMATCH", ATTEMPT_ID)
    require(persisted_evidence[0]["payload_json"] == reconciliation_json, "PUBLICATION_EVIDENCE_PAYLOAD_MISMATCH", ATTEMPT_ID)
    persisted_payload = json.loads(persisted_evidence[0]["payload_json"])
    require(persisted_payload["evidence_contract"]["schema_version"] == EVIDENCE_SCHEMA_VERSION, "PUBLICATION_EVIDENCE_PAYLOAD_SCHEMA_MISMATCH", ATTEMPT_ID)
    require(persisted_payload["evidence_contract"]["retention_days"] == EVIDENCE_RETENTION_DAYS, "PUBLICATION_EVIDENCE_RETENTION_MISMATCH", ATTEMPT_ID)
    require(persisted_payload["evidence_contract"]["max_payload_bytes"] == MAX_EVIDENCE_JSON_BYTES, "PUBLICATION_EVIDENCE_MAX_SIZE_MISMATCH", ATTEMPT_ID)
    require(persisted_payload["evidence_contract"]["canonicalization"] == "UTF-8 JSON; keys sorted; separators comma/colon; ensure_ascii=false", "PUBLICATION_EVIDENCE_CANONICALIZATION_MISMATCH", ATTEMPT_ID)
    persisted_integrity = persisted_payload["evidence_contract"]["integrity"]
    require(persisted_integrity["algorithm"] == "sha256", "PUBLICATION_EVIDENCE_INTEGRITY_ALGORITHM_MISMATCH", ATTEMPT_ID)
    require(persisted_integrity["field"] == "evidence_contract.integrity.content_sha256", "PUBLICATION_EVIDENCE_INTEGRITY_FIELD_MISMATCH", ATTEMPT_ID)
    require(persisted_integrity["verification"] == "Set content_sha256 to null, canonicalize the complete payload, then hash the UTF-8 bytes", "PUBLICATION_EVIDENCE_INTEGRITY_VERIFICATION_MISMATCH", ATTEMPT_ID)
    persisted_content_sha256 = persisted_integrity["content_sha256"]
    require(isinstance(persisted_content_sha256, str) and len(persisted_content_sha256) == 64, "PUBLICATION_EVIDENCE_CONTENT_HASH_FORMAT_INVALID", ATTEMPT_ID)
    persisted_payload["evidence_contract"]["integrity"]["content_sha256"] = None
    persisted_integrity_json = json.dumps(persisted_payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    require(hashlib.sha256(persisted_integrity_json.encode("utf-8")).hexdigest() == persisted_content_sha256, "PUBLICATION_EVIDENCE_CONTENT_HASH_MISMATCH", ATTEMPT_ID)

    assert_publication_lock_owned()
    spark.sql(
        f"""
        MERGE INTO {RELEASE_REGISTRY_TABLE} target
        USING (
            SELECT
                '{DATASET_NAME}' AS dataset_name,
                '{release_literal}' AS release_id,
                '{READY_STATE}' AS release_state,
                current_timestamp() AS reconciled_at_utc
        ) source
        ON target.dataset_name = source.dataset_name
       AND target.release_id = source.release_id
        WHEN MATCHED AND target.release_state = '{STAGING_STATE}'
         AND target.attempt_id = '{attempt_literal}'
         AND target.attempt_application_id = '{application_literal}' THEN UPDATE SET
            release_state = source.release_state,
            reconciled_at_utc = source.reconciled_at_utc,
            failure_message = NULL
        """
    )
    ready_state = (
        spark.table(RELEASE_REGISTRY_TABLE)
        .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("release_id") == RELEASE_ID))
        .select("release_state")
        .collect()[0]["release_state"]
    )
    require(ready_state == READY_STATE, "RELEASE_NOT_READY_AFTER_EVIDENCE", str(ready_state))
    ready_transition_performed = True

    if ACTIVATE_RELEASE_POINTER and not release_already_active:
        assert_publication_lock_owned()
        # This is the only activation boundary. Delta commits this one-row pointer change atomically.
        spark.sql(
            f"""
            MERGE INTO {ACTIVE_RELEASE_POINTER_TABLE} target
            USING (
                SELECT
                    '{DATASET_NAME}' AS dataset_name,
                    '{release_literal}' AS active_release_id,
                    current_timestamp() AS activated_at_utc,
                    '{sql_literal(execution_user)}' AS activated_by
            ) source
            ON target.dataset_name = source.dataset_name
            WHEN MATCHED THEN UPDATE SET
                active_release_id = source.active_release_id,
                activated_at_utc = source.activated_at_utc,
                activated_by = source.activated_by
            WHEN NOT MATCHED THEN INSERT *
            """
        )
        activation_performed = True

    if ACTIVATE_RELEASE_POINTER:
        active_pointer_after = (
            spark.table(ACTIVE_RELEASE_POINTER_TABLE)
            .filter(F.col("dataset_name") == DATASET_NAME)
            .select("active_release_id")
            .collect()
        )
        require(len(active_pointer_after) == 1, "ACTIVATION_POINTER_MISSING", DATASET_NAME)
        require(
            active_pointer_after[0]["active_release_id"] == RELEASE_ID,
            "ACTIVATION_POINTER_MISMATCH",
            str(active_pointer_after[0]["active_release_id"]),
        )
        for logical_table_name, expected_rows in EXPECTED_TABLE_ROWS.items():
            assert_publication_lock_owned()
            active_view = spark.table(logical_table_name)
            actual_rows = active_view.count()
            require(actual_rows == expected_rows, "ACTIVE_VIEW_ROW_COUNT_MISMATCH", f"{logical_table_name} expected={expected_rows} actual={actual_rows}")
            require_unique(active_view, TABLE_KEYS[logical_table_name], f"ACTIVE_VIEW_{logical_table_name.upper().replace('.', '_')}_KEY_VIOLATION")
            active_view_hash = deterministic_content_hash(
            active_view,
            logical_table_name=logical_table_name,
            expected_business_columns=EXPECTED_BUSINESS_COLUMNS[logical_table_name],
            schema_error_code="ACTIVE_VIEW_CONTENT_HASH_SCHEMA_DRIFT",
        )
        require(
            active_view_hash == prepublication_hashes[logical_table_name],
            "ACTIVE_VIEW_CONTENT_HASH_MISMATCH",
            json.dumps(
                {
                    "logical_table": logical_table_name,
                    "expected_sha256": prepublication_hashes[logical_table_name],
                    "actual_sha256": active_view_hash,
                    "business_columns": EXPECTED_BUSINESS_COLUMNS[logical_table_name],
                },
                sort_keys=True,
            ),
        )
except Exception as evidence_error:
    evidence_failure_type = sanitized_exception_type(evidence_error)
    attempt_mark_failure_type = "NONE"
    try:
        if ACTIVATE_RELEASE_POINTER and ready_transition_performed and not activation_performed:
            activation_failure_literal = sql_literal(f"ACTIVATION_AFTER_READY_FAILED:{evidence_failure_type}")
            spark.sql(
                f"""
                UPDATE {RELEASE_REGISTRY_TABLE}
                SET release_state = '{FAILED_STATE}',
                    failure_message = '{activation_failure_literal}'
                WHERE dataset_name = '{DATASET_NAME}'
                  AND release_id = '{release_literal}'
                  AND attempt_id = '{attempt_literal}'
                """
            )
        evidence_failure_literal = sql_literal(f"EVIDENCE_PERSISTENCE_FAILED:{evidence_failure_type}")
        spark.sql(
            f"""
            UPDATE {PUBLICATION_ATTEMPT_TABLE}
            SET attempt_state = '{FAILED_STATE}', completed_at_utc = current_timestamp(),
                failure_message = '{evidence_failure_literal}'
            WHERE dataset_name = '{DATASET_NAME}'
              AND attempt_id = '{attempt_literal}'
            """
        )
        failed_attempt = (
            spark.table(PUBLICATION_ATTEMPT_TABLE)
            .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("attempt_id") == ATTEMPT_ID))
            .select("attempt_state", "failure_message")
            .collect()
        )
        require(
            len(failed_attempt) == 1
            and failed_attempt[0]["attempt_state"] == FAILED_STATE
            and failed_attempt[0]["failure_message"] == f"EVIDENCE_PERSISTENCE_FAILED:{evidence_failure_type}",
            "EVIDENCE_ATTEMPT_FAILURE_MARK_FAILED",
            ATTEMPT_ID,
        )
    except Exception as attempt_mark_error:
        attempt_mark_failure_type = sanitized_exception_type(attempt_mark_error)
    raise RuntimeError(
        "EVIDENCE_PERSISTENCE_FAILED_LOCK_RETAINED: "
        f"evidence_error_type={evidence_failure_type}; "
        f"attempt_mark_error_type={attempt_mark_failure_type}"
    ) from evidence_error

spark.sql(
    f"""
    UPDATE {PUBLICATION_ATTEMPT_TABLE}
    SET attempt_state = 'SUCCEEDED', completed_at_utc = current_timestamp()
    WHERE dataset_name = '{DATASET_NAME}'
      AND attempt_id = '{attempt_literal}'
    """
)
completed_attempt = (
    spark.table(PUBLICATION_ATTEMPT_TABLE)
    .filter((F.col("dataset_name") == DATASET_NAME) & (F.col("attempt_id") == ATTEMPT_ID))
    .select("attempt_state")
    .collect()
)
require(len(completed_attempt) == 1 and completed_attempt[0]["attempt_state"] == "SUCCEEDED", "PUBLICATION_ATTEMPT_COMPLETION_FAILED_LOCK_RETAINED", ATTEMPT_ID)
assert_publication_lock_owned()
spark.sql(
    f"""
    UPDATE {PUBLICATION_LOCK_TABLE}
    SET attempt_id = NULL,
        attempt_application_id = NULL,
        acquired_at_utc = NULL
    WHERE dataset_name = '{DATASET_NAME}'
      AND attempt_id = '{attempt_literal}'
      AND attempt_application_id = '{application_literal}'
    """
)
require(
    spark.table(PUBLICATION_LOCK_TABLE).filter((F.col("dataset_name") == DATASET_NAME) & (F.col("attempt_id") == ATTEMPT_ID)).count() == 0,
    "PUBLICATION_LOCK_RELEASE_FAILED_OPERATOR_CLEANUP_REQUIRED",
    ATTEMPT_ID,
)

# Display mirrors the already persisted payload and is not evidence authority.
print(reconciliation_json)
display(spark.createDataFrame([Row(reconciliation_json=reconciliation_json)]))


# METADATA ********************

# META {
# META   "language": "python",
# META   "language_group": "synapse_pyspark"
# META }
